# Cuaderno 8 — Capa 3: modelo de detección y evaluación

Este cuaderno reemplaza al 07. Corre de arriba abajo sin depender de resultados previos de la Capa 3:
lee las bases ya construidas (`base_maestra_v4`, `oferentes_consolidado`, `directorio_companias`,
catastro del SRI) y **escribe todo con el prefijo `c8_`**, de modo que nada de lo anterior se sobrescribe.

**Orden corregido respecto del cuaderno 07.** Allá la exploración de la Superintendencia corría después
del modelo y había que parchear el panel hacia atrás. Aquí las variables societarias se construyen antes
del panel, y el panel se arma una sola vez con todo adentro.

| Paso | Qué hace | Qué escribe |
|---|---|---|
| 1 | Entorno, vistas, chequeos de integridad, utilidades | `c8_manifiesto.json` |
| 2 | Etiqueta de fantasma con fecha de reactivación | `c8_etiqueta.parquet` |
| 3 | Variables de la Superintendencia (solo antecedentes legítimos) | `c8_supercias.parquet` |
| 4 | Índice de riesgo tipo CRI, sin Benford | `c8_cri_proceso.parquet`, `c8_cri_prov_anio.parquet` |
| 5 | Panel de modelado proveedor × año, con las dos etiquetas | `c8_panel.parquet` |
| 6 | Auditoría de fuga y conjuntos de variables | `c8_vars.json`, `c8_auditoria.csv` |
| 7 | Modelo PU con bagging y evaluación por conglomerado | `c8_resultados.csv`, `c8_scores.parquet` |
| 8 | Ablación por familia de variables | `c8_ablacion.csv` |
| 9 | Alerta temprana: solo contratos anteriores a la marca del SRI | `c8_alerta_temprana.csv` |
| 10 | Lista priorizada de proveedores para revisión | `c8_priorizacion.csv` |

**Reglas que no se negocian.** Cero fuga temporal: las variables de red usan solo años anteriores.
Nada de imputar: los faltantes van como tales y el modelo los maneja. La fuga se decide por el origen
de la variable, no por su AUC. Los positivos del SRI validan; no son el único insumo.


## Paso 1 — Entorno, integridad y utilidades

Correr después de **cada** reinicio del kernel. El resto de celdas lee de disco, así que basta con esta.

In [ ]:
# ============ PASO 1 · Entorno, integridad y utilidades ============
import os, re, sys, gc, io, json, time, glob, contextlib, unicodedata, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 80)

os.chdir(r'C:\SERCOP_FINAL')
R, D = '03_resultados', '05_documentos'
OUT = f'{R}/c8_'                      # TODO lo que escribe este cuaderno lleva este prefijo
SEMILLA = 20261001
rng = np.random.default_rng(SEMILLA)

con = duckdb.connect(); con.execute("PRAGMA threads=8")
print('Carpeta:', os.getcwd())
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| duckdb', duckdb.__version__, '| numpy', np.__version__)

# ---- entradas: no se modifican nunca ----
ENTRADAS = {'bm4':'base_maestra_v4.parquet', 'ofe':'oferentes_consolidado.parquet',
            'dir':'directorio_companias.parquet'}
print('\n--- Entradas ---')
for v, arch in ENTRADAS.items():
    ruta = f'{R}/{arch}'
    assert os.path.exists(ruta), f'FALTA {ruta}'
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM read_parquet('{ruta}')")
    n = con.execute(f"SELECT COUNT(*) FROM {v}").fetchone()[0]
    k = len(con.execute(f"DESCRIBE {v}").fetchall())
    mt = time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(ruta)))
    print(f'  {v:4s} {n:>10,} filas x {k:>3} cols | {mt} | {arch}')

# vista canonica: el RUC vacio agrupa 42.294 filas y $2.338M bajo un proveedor inexistente
con.execute("""CREATE OR REPLACE VIEW bmv AS SELECT * FROM bm4
               WHERE proveedor_ruc IS NOT NULL AND TRIM(proveedor_ruc) <> ''""")

print('\n--- Chequeos ---')
def chk(nombre, obtenido, esperado):
    ok = obtenido == esperado
    print(f'  {"OK   " if ok else "FALLA"} {nombre}: {obtenido:,} (esperado {esperado:,})')
    return ok
todo = True
todo &= chk('filas bm4', con.execute("SELECT COUNT(*) FROM bm4").fetchone()[0], 2736387)
todo &= chk('filas bmv', con.execute("SELECT COUNT(*) FROM bmv").fetchone()[0], 2690017)
todo &= chk('RUC fantasma en bmv',
            con.execute("SELECT COUNT(DISTINCT proveedor_ruc) FROM bmv WHERE proveedor_es_fantasma").fetchone()[0], 15)
print('\n', 'TODO EN ORDEN' if todo else '!! REVISAR ANTES DE SEGUIR')

# ---- utilidades comunes ----
def norm_nombre(s):
    """Nombre de persona normalizado: sin tildes, mayusculas, solo letras. None si no sirve."""
    if pd.isna(s): return None
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode('ascii').upper()
    s = re.sub(r'[^A-Z ]',' ',s); s = re.sub(r'\s+',' ',s).strip()
    return s if len(s) >= 8 and len(s.split()) >= 2 else None

def a_fecha(s):
    """Acepta numero de serie de Excel (43003) o texto (20/07/1951)."""
    n = pd.to_numeric(s, errors='coerce')
    f = pd.to_datetime(n, unit='D', origin='1899-12-30', errors='coerce')
    return f.fillna(pd.to_datetime(s, errors='coerce', dayfirst=True))

def auc(x, y, minimo=20):
    """AUC de una variable sola. Devuelve NaN si no hay datos suficientes."""
    x = np.asarray(x, dtype=float); y = np.asarray(y)
    m = ~np.isnan(x)
    if m.sum() < minimo or y[m].sum() == 0 or y[m].sum() == m.sum(): return np.nan
    r = pd.Series(x[m]).rank().values; yy = y[m]
    n1, n0 = yy.sum(), len(yy) - yy.sum()
    return (r[yy == 1].sum() - n1*(n1+1)/2) / (n1*n0)

def guardar(df, nombre):
    """Escribe a temporal y recien reemplaza: si algo falla, no te quedas sin el archivo anterior."""
    tmp = f'{OUT}{nombre}.tmp.parquet'
    df.to_parquet(tmp, index=False); os.replace(tmp, f'{OUT}{nombre}.parquet')
    print(f'  -> {OUT}{nombre}.parquet ({len(df):,} filas x {df.shape[1]} cols)')

MANIFIESTO = {'semilla': SEMILLA, 'inicio': time.strftime('%Y-%m-%d %H:%M:%S'),
              'python': sys.version.split()[0], 'pandas': pd.__version__,
              'numpy': np.__version__, 'duckdb': duckdb.__version__, 'pasos': {}}
def anotar(paso, **kw):
    MANIFIESTO['pasos'][paso] = dict(kw, hora=time.strftime('%H:%M:%S'))
    with open(f'{OUT}manifiesto.json','w',encoding='utf-8') as f:
        json.dump(MANIFIESTO, f, ensure_ascii=False, indent=1)

print('\nUtilidades: norm_nombre(), a_fecha(), auc(), guardar(), anotar()')
print(f'Prefijo de salida: {OUT}')

## Paso 2 — Etiqueta de empresa fantasma, con fecha

El catastro del SRI no es una marca permanente: 33 de los 1.040 RUC fueron reactivados. Un proceso
posterior a la reactivación **no** cuenta como positivo. La columna `momento` clasifica cada contrato
respecto de la calificación, y es la que sostiene el argumento de alerta temprana: la mayoría de los
contratos de estas empresas ocurrió **antes** de que el SRI las marcara.

In [ ]:
# ===================================================================
# PERFIL del catastro del RUC del SRI, antes de cruzarlo con nada
# Dice qué trae, cuánto cruza, y qué columnas son usables sin fuga.
# ===================================================================
import os, re, glob, unicodedata, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 60)
R, D = '03_resultados', '05_documentos'

# ---------- 1) Encontrar el archivo ----------
pats = ['*atastro*RUC*', '*RUC*atastro*', '*catastro*', '*ruc*']
cand = []
for carp in [R, D, '.', '04_datos', '02_datos']:
    for p in pats:
        for ext in ('.csv','.txt','.parquet','.xlsx'):
            cand += glob.glob(os.path.join(carp, p + ext))
            cand += glob.glob(os.path.join(carp, '**', p + ext), recursive=True)
cand = sorted({c for c in cand if 'antasma' not in c})          # el de fantasmas ya lo usamos
print('Candidatos encontrados:')
for c in cand: print(f'  {os.path.getsize(c)/1e6:9.1f} MB  {c}')
assert cand, ('No encuentro el catastro del RUC. Pon la ruta a mano en ARCHIVO = r"..." '
              'y vuelve a correr desde la línea siguiente.')
ARCHIVO = max(cand, key=os.path.getsize)        # el más grande suele ser el consolidado
print(f'\nSe usará: {ARCHIVO}')

# ---------- 2) Leerlo adivinando separador y codificación ----------
def leer(ruta, n=None):
    if ruta.endswith('.parquet'): return pd.read_parquet(ruta)
    if ruta.endswith('.xlsx'):    return pd.read_excel(ruta, nrows=n)
    for sep in ['|', ';', '\t', ',']:
        for enc in ['latin-1', 'utf-8']:
            try:
                d = pd.read_csv(ruta, sep=sep, encoding=enc, dtype=str, nrows=n or 5000,
                                on_bad_lines='skip', low_memory=False)
                if d.shape[1] >= 4:
                    print(f'  separador "{sep}" | codificación {enc} | {d.shape[1]} columnas')
                    return pd.read_csv(ruta, sep=sep, encoding=enc, dtype=str,
                                       on_bad_lines='skip', low_memory=False) if n is None else d
            except Exception: pass
    raise RuntimeError('No pude leer el archivo: revisar separador y codificación a mano')
ruc = leer(ARCHIVO)
ruc.columns = [c.strip().upper() for c in ruc.columns]
print(f'\nFilas: {len(ruc):,} | columnas: {ruc.shape[1]}')
print('Columnas:', list(ruc.columns))

# ---------- 3) Perfil de cada columna ----------
print('\n' + '='*66 + '\nPERFIL DE COLUMNAS\n' + '='*66)
for c in ruc.columns:
    nn = ruc[c].notna().sum(); card = ruc[c].nunique(dropna=True)
    print(f'\n{c}  ({100*nn/len(ruc):.1f}% con dato, {card:,} distintos)')
    if 0 < card <= 20: print('   ', ruc[c].value_counts().head(10).to_dict())
    else:              print('    ejemplos:', ruc[c].dropna().astype(str).head(3).tolist())

# ---------- 4) ¿Cuánto cruza con nuestros proveedores? ----------
c_ruc = next((c for c in ruc.columns if 'RUC' in c and ruc[c].astype(str).str.len().median() >= 10), None)
print('\n' + '='*66 + f'\nCRUCE  (columna de RUC detectada: {c_ruc})\n' + '='*66)
assert c_ruc, 'No identifico la columna de RUC: revisar los nombres de arriba'
ruc['ruc13'] = ruc[c_ruc].astype(str).str.replace(r'\D','',regex=True).str.zfill(13)
ruc = ruc.drop_duplicates(subset='ruc13')
con.register('ruc_t', ruc[['ruc13']])
print(con.execute("""
  SELECT COUNT(DISTINCT b.proveedor_ruc) AS proveedores_sercop,
         COUNT(DISTINCT CASE WHEN r.ruc13 IS NOT NULL THEN b.proveedor_ruc END) AS cruzan,
         ROUND(100.0*COUNT(DISTINCT CASE WHEN r.ruc13 IS NOT NULL THEN b.proveedor_ruc END)
               / COUNT(DISTINCT b.proveedor_ruc),1) AS pct
  FROM bmv b LEFT JOIN ruc_t r ON r.ruc13 = b.proveedor_ruc""").df().to_string(index=False))
print('\nCobertura por tipo de contribuyente (tercer dígito del RUC), para ver a quién alcanza:')
print(con.execute("""
  SELECT CASE WHEN SUBSTR(b.proveedor_ruc,3,1) IN ('0','1','2','3','4','5') THEN 'persona natural'
              WHEN SUBSTR(b.proveedor_ruc,3,1)='6' THEN 'institucion publica'
              WHEN SUBSTR(b.proveedor_ruc,3,1)='9' THEN 'sociedad privada' ELSE 'otro' END AS tipo,
         COUNT(DISTINCT b.proveedor_ruc) AS proveedores,
         ROUND(100.0*COUNT(DISTINCT CASE WHEN r.ruc13 IS NOT NULL THEN b.proveedor_ruc END)
               / COUNT(DISTINCT b.proveedor_ruc),1) AS pct_cruza
  FROM bmv b LEFT JOIN ruc_t r ON r.ruc13 = b.proveedor_ruc GROUP BY 1 ORDER BY 2 DESC""").df().to_string(index=False))

# ---------- 5) Clasificar columnas: antecedente, consecuencia o con fecha ----------
print('\n' + '='*66 + '\nQUÉ SE PUEDE USAR Y QUÉ NO\n' + '='*66)
REGLAS = [
 ('ANTECEDENTE', 'entra al modelo', ('CIIU','ACTIVIDAD','INICIO_ACTIVIDADES','FECHA_INICIO','PROVINCIA',
                                     'CANTON','PARROQUIA','TIPO_CONTRIBUYENTE','CLASE_CONTRIBUYENTE',
                                     'ESTABLECIMIENTOS','OBLIGADO','CONTABILIDAD','RAZON_SOCIAL','NOMBRE')),
 ('CONSECUENCIA','NO entra (efecto de la calificación)', ('ESTADO_CONTRIBUYENTE','ESTADO_RUC','SUSPENSION',
                                     'ESTADO_ESTABLECIMIENTO','MOTIVO','CESACION','CANCELACION')),
 ('CON FECHA',   'usable SOLO si se compara contra la fecha del contrato',
                                    ('FECHA_SUSPENSION','FECHA_CESACION','FECHA_REINICIO','FECHA_ACTUALIZACION')),
]
clasif = []
for c in ruc.columns:
    if c == 'ruc13': continue
    etiqueta, nota = 'REVISAR A MANO', 'no reconocida: decidir según su origen'
    for e, n, pats_ in REGLAS:
        if any(p in c for p in pats_): etiqueta, nota = e, n; break
    clasif.append((c, etiqueta, nota, f'{100*ruc[c].notna().mean():.0f}%'))
cl = pd.DataFrame(clasif, columns=['columna','clasificacion','nota','cobertura'])
for e in ['ANTECEDENTE','CON FECHA','CONSECUENCIA','REVISAR A MANO']:
    s = cl[cl.clasificacion == e]
    if len(s): print(f'\n--- {e} ---'); print(s[['columna','cobertura','nota']].to_string(index=False))

# ---------- 6) ¿Hay fecha de suspensión? ----------
f_sus = [c for c in ruc.columns if 'SUSPEN' in c and 'FECHA' in c]
print(f'\nColumnas de fecha de suspensión: {f_sus if f_sus else "NINGUNA"}')
print('Con fecha de suspensión se puede construir "contrató con el RUC suspendido ese día",')
print('que es una irregularidad dura y NO es fuga. Sin fecha, el estado del RUC queda fuera del modelo.')

ruc.to_parquet(f'{R}/c8_catastro_ruc.parquet', index=False)
print(f'\n  -> {R}/c8_catastro_ruc.parquet ({len(ruc):,} filas)')

In [ ]:
# ===================================================================
# RUC-A · Reconocer el esquema del catastro del SRI (sin cargar nada grande)
# ===================================================================
import os, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 240); pd.set_option('display.max_columns', 60)
CARPETA = r'05_documentos\SRI'

arch = sorted(glob.glob(os.path.join(CARPETA, '*.csv')) + glob.glob(os.path.join(CARPETA, '*.txt')))
assert arch, f'No hay archivos en {CARPETA}'
print(f'Archivos: {len(arch)} | total {sum(os.path.getsize(a) for a in arch)/1e9:.2f} GB')
for a in sorted(arch, key=os.path.getsize)[:3] + sorted(arch, key=os.path.getsize)[-2:]:
    print(f'  {os.path.getsize(a)/1e6:9.1f} MB  {os.path.basename(a)}')

chico = min(arch, key=os.path.getsize)
print(f'\nSe analiza el más pequeño: {os.path.basename(chico)}')
with open(chico, 'rb') as f: crudo = f.read(3000)
print('\nPrimeros bytes:'); print(repr(crudo[:300]))

OK = None
for sep in ['|', ';', ',', '\t']:
    for enc in ['latin-1', 'utf-8', 'cp1252']:
        try:
            q = (f"SELECT * FROM read_csv('{chico.replace(chr(92),'/')}', delim='{sep}', "
                 f"header=true, all_varchar=true, encoding='{enc}', ignore_errors=true) LIMIT 5")
            d = con.execute(q).df()
            if d.shape[1] >= 5:
                print(f'\nFUNCIONA -> delim="{sep}" encoding="{enc}" | {d.shape[1]} columnas'); OK = (sep, enc, d); break
        except Exception:
            pass
    if OK: break
assert OK, 'Ninguna combinación funcionó: abre un archivo en el Bloc de notas y dime qué ves'
SEP, ENC, muestra = OK
print('\nColumnas:'); print(list(muestra.columns))
print('\nPrimeras filas:'); print(muestra.to_string(index=False, max_colwidth=22))

print('\nVerificando que las 24 provincias tengan las mismas columnas...')
base = list(muestra.columns); distintos = []
for a in arch:
    try:
        c = list(con.execute(f"SELECT * FROM read_csv('{a.replace(chr(92),'/')}', delim='{SEP}', "
                             f"header=true, all_varchar=true, encoding='{ENC}', ignore_errors=true) LIMIT 1").df().columns)
        if c != base: distintos.append((os.path.basename(a), len(c)))
    except Exception as e: distintos.append((os.path.basename(a), f'ERROR {e}'))
print('  Todos iguales' if not distintos else f'  !! Distintos: {distintos}')
print(f'\nGuarda estos valores para la celda siguiente:  SEP = "{SEP}"   ENC = "{ENC}"')

In [ ]:
# ===================================================================
# DIAGNÓSTICO de las observaciones al anteproyecto
# ¿se imputaron constantes? ¿el indicador de proveedor único deja fuera
# los selectivos? ¿cuánto pesa el catálogo electrónico?
# ===================================================================
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
R = '03_resultados'
cols = [c[0] for c in con.execute("DESCRIBE bmv").fetchall()]

# ---------- 1) ¿Quedó imputación de constantes en la base? ----------
print('='*66 + '\n1. Valores ausentes rellenados con constantes\n' + '='*66)
c_of = [c for c in cols if 'oferente' in c.lower() and c.startswith('n_')] or \
       [c for c in cols if c in ('n_oferentes','num_oferentes','cantidad_oferentes')]
c_pz = [c for c in cols if any(k in c.lower() for k in ('plazo','duracion','dias'))]
print('Columnas de número de oferentes:', c_of, '| de plazo/duración:', c_pz)
for c in c_of + c_pz:
    d = con.execute(f'''SELECT COUNT(*) AS filas,
            SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END) AS nulos,
            SUM(CASE WHEN "{c}" = 0 THEN 1 ELSE 0 END) AS igual_0,
            SUM(CASE WHEN "{c}" = 1 THEN 1 ELSE 0 END) AS igual_1,
            ROUND(AVG(CAST("{c}" AS DOUBLE)),2) AS media
        FROM bmv''').df()
    d.insert(0, 'columna', c); print(d.to_string(index=False))

# prueba decisiva: para procesos QUE SÍ están en la tabla de oferentes,
# ¿el número declarado coincide con el número realmente observado?
if c_of:
    c0 = c_of[0]
    print(f'\n¿"{c0}" coincide con el conteo real de la tabla de oferentes?')
    print(con.execute(f'''
      WITH real AS (SELECT ocid, COUNT(DISTINCT oferente_ruc) AS n_real FROM ofe
                    WHERE TRIM(COALESCE(oferente_ruc,'')) <> '' GROUP BY 1)
      SELECT CASE WHEN r.ocid IS NULL THEN 'sin datos de oferentes' ELSE 'con datos' END AS situacion,
             COUNT(*) AS filas, ROUND(AVG(CAST(b."{c0}" AS DOUBLE)),2) AS declarado_medio,
             ROUND(AVG(r.n_real),2) AS real_medio,
             ROUND(100.0*AVG(CASE WHEN b."{c0}" = r.n_real THEN 1 ELSE 0 END),1) AS pct_coincide,
             ROUND(100.0*AVG(CASE WHEN b."{c0}" = 1 THEN 1 ELSE 0 END),1) AS pct_declarado_en_1
      FROM (SELECT DISTINCT ocid, "{c0}" FROM bmv) b LEFT JOIN real r ON r.ocid = b.ocid
      GROUP BY 1 ORDER BY 2 DESC''').df().to_string(index=False))
    print('Si en "sin datos de oferentes" el declarado medio es 1.00, la constante sigue en la base.')

# ---------- 2) Proveedor único por tipo de contratación ----------
print('\n' + '='*66 + '\n2. Proveedor único por tipo de contratación\n' + '='*66)
t = con.execute('''
  SELECT tipo_contratacion,
         COUNT(DISTINCT ocid) AS procesos,
         ROUND(100.0*AVG(CASE WHEN proveedor_unico THEN 1 ELSE 0 END),1) AS pct_unico,
         ROUND(100.0*AVG(CASE WHEN tiene_periodo_competitivo THEN 1 ELSE 0 END),1) AS pct_marcado_competitivo,
         ROUND(SUM(monto_final)/1e6,1) AS monto_millones
  FROM bmv GROUP BY 1 ORDER BY procesos DESC''').df()
print(t.head(25).to_string(index=False))
fuera = con.execute('''
  SELECT COUNT(DISTINCT ocid) AS procesos, ROUND(SUM(monto_final)/1e6,1) AS monto_millones
  FROM bmv WHERE proveedor_unico AND NOT tiene_periodo_competitivo''').df()
print('\nProcesos con UN SOLO oferente que hoy quedan FUERA del universo competitivo:')
print(fuera.to_string(index=False))
print('El evaluador menciona 31.287 casos de tipo selectivo. Comparar con esta cifra.')

# ---------- 3) Peso del catálogo electrónico ----------
print('\n' + '='*66 + '\n3. Catálogo electrónico y régimen normativo\n' + '='*66)
cat = con.execute('''
  SELECT CASE WHEN UPPER(tipo_contratacion) LIKE '%CAT_LOGO%'
               OR UPPER(tipo_contratacion) LIKE '%CATALOGO%' THEN 'catalogo electronico'
              ELSE 'resto' END AS universo,
         COUNT(*) AS filas, COUNT(DISTINCT ocid) AS procesos,
         COUNT(DISTINCT proveedor_ruc) AS proveedores,
         ROUND(SUM(monto_final)/1e6,1) AS monto_millones,
         ROUND(MEDIAN(monto_final),0) AS monto_mediano,
         ROUND(100.0*SUM(monto_final)/(SELECT SUM(monto_final) FROM bmv),1) AS pct_dinero
  FROM bmv GROUP BY 1 ORDER BY filas DESC''').df()
print(cat.to_string(index=False))
print('\nRegímenes que desaparecen en 2026 (Menor Cuantía y Cotización):')
print(con.execute('''
  SELECT tipo_contratacion, COUNT(DISTINCT ocid) AS procesos,
         MIN(anio_proceso) AS desde, MAX(anio_proceso) AS hasta,
         ROUND(SUM(monto_final)/1e6,1) AS monto_millones
  FROM bmv WHERE UPPER(tipo_contratacion) LIKE '%MENOR CUANT%'
             OR UPPER(tipo_contratacion) LIKE '%COTIZACI%'
  GROUP BY 1 ORDER BY procesos DESC''').df().to_string(index=False))
print('\nCifras clave CON y SIN catálogo electrónico:')
print(con.execute('''
  WITH b AS (SELECT *, (UPPER(tipo_contratacion) LIKE '%CAT_LOGO%'
                     OR UPPER(tipo_contratacion) LIKE '%CATALOGO%') AS es_cat FROM bmv)
  SELECT 'con catalogo' AS universo, COUNT(*) AS filas,
         COUNT(DISTINCT proveedor_ruc) AS proveedores,
         ROUND(SUM(monto_final)/1e6,1) AS millones,
         ROUND(100.0*AVG(CASE WHEN proveedor_unico THEN 1 ELSE 0 END),1) AS pct_unico FROM b
  UNION ALL
  SELECT 'sin catalogo', COUNT(*), COUNT(DISTINCT proveedor_ruc),
         ROUND(SUM(monto_final)/1e6,1), ROUND(100.0*AVG(CASE WHEN proveedor_unico THEN 1 ELSE 0 END),1)
  FROM b WHERE NOT es_cat''').df().to_string(index=False))

# ---------- 4) Cadena de filtros, para documentarla ----------
print('\n' + '='*66 + '\n4. Cadena de filtros (para la sección de datos)\n' + '='*66)
print(con.execute('''
  SELECT 'base_maestra_v4 completa' AS paso, COUNT(*) AS filas FROM bm4
  UNION ALL SELECT 'menos RUC vacio o nulo', COUNT(*) FROM bmv
  UNION ALL SELECT 'menos catalogo electronico', COUNT(*) FROM bmv
     WHERE NOT (UPPER(tipo_contratacion) LIKE '%CAT_LOGO%' OR UPPER(tipo_contratacion) LIKE '%CATALOGO%')
  UNION ALL SELECT 'menos 2026 (anio incompleto)', COUNT(*) FROM bmv
     WHERE NOT (UPPER(tipo_contratacion) LIKE '%CAT_LOGO%' OR UPPER(tipo_contratacion) LIKE '%CATALOGO%')
       AND anio_proceso < 2026''').df().to_string(index=False))
print('\nFecha de corte de la descarga (máxima fecha observada):')
c_f = next((c for c in cols if c in ('fecha_release','fecha_adjudicacion','fecha_publicacion')), None)
if c_f: print(con.execute(f'SELECT MAX(TRY_CAST("{c_f}" AS TIMESTAMP)) AS ultima_fecha FROM bmv').df().to_string(index=False))

In [ ]:
# ============ PASO 2 · Etiqueta de fantasma con fecha ============
cand = glob.glob(f'{D}/*antasma*.csv') + glob.glob(f'{D}/**/*antasma*.csv', recursive=True)
assert cand, f'No encuentro el catastro del SRI en {D}'
fx = pd.read_csv(cand[0], sep='|', encoding='latin-1', dtype=str)
fx.columns = [c.strip() for c in fx.columns]          # el archivo trae ' RAZON_SOCIAL' con espacio
print(f'Catastro: {cand[0]} | {len(fx):,} filas')

MAPA = {'f_oficio':'FECHA_NOTIFICACION_OFICIO', 'f_resolucion':'FECHA_NOTIFICACION_RESOLUCION',
        'f_inicio_cal':'FECHA_INICIO_CALIFICACION', 'f_fin_cal':'FECHA_FIN_CALIFICACION',
        'f_react_notif':'FECHA_NOTIFICACION_REACTIVACION', 'f_react_ruc':'FECHA_REACTIVACION_RUC'}
for k, c in MAPA.items():
    fx[k] = a_fecha(fx[c]) if c in fx.columns else pd.NaT
    print(f'  {k:14s} <- {c:35s} {fx[k].notna().sum():>5,} con fecha')

fx['ruc'] = fx['NUMERO_RUC'].str.replace(r'\.0$','',regex=True).str.strip().str.zfill(13)
fx['f_reactivacion'] = fx[['f_react_notif','f_react_ruc']].min(axis=1)
sri = (fx.groupby('ruc')
         .agg(f_oficio=('f_oficio','min'), f_resolucion=('f_resolucion','min'),
              f_inicio_cal=('f_inicio_cal','min'), f_fin_cal=('f_fin_cal','max'),
              f_reactivacion=('f_reactivacion','max')).reset_index())
sri['f_marca'] = sri[['f_oficio','f_resolucion','f_inicio_cal']].min(axis=1)
print(f'\nRUC en el catastro: {len(sri):,} | con reactivacion: {sri.f_reactivacion.notna().sum():,}')

cols_bmv = [c[0] for c in con.execute("DESCRIBE bmv").fetchall()]
c_fecha = next((c for c in ['fecha_adjudicacion','fecha_publicacion','fecha_release',
                            'fecha_release_json','fecha'] if c in cols_bmv), None)
fecha_sql = f'TRY_CAST("{c_fecha}" AS TIMESTAMP)' if c_fecha else 'NULL'
print('Fecha del proceso tomada de:', c_fecha)

con.register('sri_t', sri[['ruc','f_marca','f_fin_cal','f_reactivacion']])
etq = con.execute(f"""
WITH p AS (SELECT ocid, proveedor_ruc, anio_proceso,
                  COALESCE({fecha_sql}, MAKE_TIMESTAMP(anio_proceso,12,31,0,0,0)) AS fecha_proc
           FROM bmv)
SELECT p.ocid, p.proveedor_ruc, p.anio_proceso, p.fecha_proc,
       (s.ruc IS NOT NULL) AS en_catastro, s.f_marca, s.f_reactivacion,
       (s.ruc IS NOT NULL AND (s.f_reactivacion IS NULL OR p.fecha_proc < s.f_reactivacion)) AS y_fantasma,
       CASE WHEN s.ruc IS NULL THEN NULL
            WHEN s.f_marca IS NOT NULL AND p.fecha_proc <  s.f_marca THEN 'antes de la marca'
            WHEN s.f_reactivacion IS NOT NULL AND p.fecha_proc >= s.f_reactivacion THEN 'tras reactivacion'
            ELSE 'durante la calificacion' END AS momento
FROM p LEFT JOIN sri_t s ON s.ruc = p.proveedor_ruc""").df()

print('\n--- Etiqueta ---')
print(f"RUC en catastro: {etq.loc[etq.en_catastro,'proveedor_ruc'].nunique()} | "
      f"RUC positivos: {etq.loc[etq.y_fantasma,'proveedor_ruc'].nunique()} | "
      f"filas positivas: {int(etq.y_fantasma.sum())}")
print('\nMomento de los contratos de empresas del catastro:')
print(etq[etq.en_catastro].groupby('momento').agg(filas=('ocid','size'),
      ruc=('proveedor_ruc','nunique')).sort_values('filas', ascending=False).to_string())
guardar(etq, 'etiqueta')
anotar('2_etiqueta', ruc_positivos=int(etq.loc[etq.y_fantasma,'proveedor_ruc'].nunique()),
       filas_positivas=int(etq.y_fantasma.sum()))

## Paso 3 — Variables de la Superintendencia

Dos categorías, y la distinción es el centro del asunto:

- **Antecedentes** (entran al modelo): fecha de constitución, tipo societario, provincia. Existen antes
  del contrato y son independientes del registro del SRI.
- **Consecuencias** (quedan fuera): situación legal y cargo del representante. El 100% de las empresas
  calificadas como fantasma figura hoy en disolución o liquidación, frente al 6,7% del resto. Eso es el
  efecto administrativo de la calificación, medido hoy y aplicado hacia atrás. Usarlo sería circular.

Las indicadoras quedan **nulas** para quien no está en el directorio, no en cero: un cero diría
"no es sociedad anónima" cuando en realidad quiere decir "no es una sociedad".

In [ ]:
# ============ PASO 3 · Variables de la Superintendencia ============
dr = con.execute("SELECT * FROM dir").df()
dr.columns = [c.strip() for c in dr.columns]
dr = dr.drop_duplicates(subset='RUC')
print(f'Directorio: {len(dr):,} RUC | columnas: {list(dr.columns)}')

col = lambda *pats: next((c for c in dr.columns if any(p in c.upper() for p in pats)), None)
c_sit, c_tip = col('SITUAC'), col('TIPO')
c_con, c_pro, c_car = col('CONSTITUC'), col('PROVINCIA'), col('CARGO')

sup = pd.DataFrame({'proveedor_ruc': dr.RUC.astype(str)})
sup['sup_anio_constitucion'] = a_fecha(dr[c_con]).dt.year if c_con else np.nan
sup['sup_tipo']      = dr[c_tip].map(norm_nombre if False else (lambda s: None if pd.isna(s) else str(s).strip().upper())) if c_tip else None
sup['sup_provincia'] = dr[c_pro].map(lambda s: None if pd.isna(s) else str(s).strip().upper()) if c_pro else None
sup['rep_norm']      = dr['REPRESENTANTE'].map(norm_nombre) if 'REPRESENTANTE' in dr else None

# solo para documentar: NO entran al modelo
if c_sit:
    s = dr[c_sit].map(lambda x: None if pd.isna(x) else str(x).strip().upper())
    sup['doc_no_activa'] = (~s.fillna('').str.contains('ACTIVA')).astype(float)
print(f'\nFecha de constitucion valida en {sup.sup_anio_constitucion.notna().mean()*100:.1f}% de las empresas')

# ---- parte proveedor-anio: antiguedad y dispersion geografica ----
con.register('sup_t', sup[['proveedor_ruc','sup_anio_constitucion','sup_provincia']])
c_ep = next((c for c in [c[0] for c in con.execute("DESCRIBE bmv").fetchall()]
             if 'provincia' in c.lower()), None)
print('Provincia de la entidad:', c_ep)
pa = con.execute(f"""
SELECT b.proveedor_ruc, b.anio_proceso,
       MAX(b.anio_proceso - s.sup_anio_constitucion) AS sup_antiguedad,
       {f'AVG(CASE WHEN b."{c_ep}" IS NULL THEN NULL WHEN UPPER(TRIM(b."{c_ep}")) <> s.sup_provincia THEN 1.0 ELSE 0.0 END)'
        if c_ep else 'NULL'} AS sup_frac_fuera_provincia
FROM bmv b JOIN sup_t s ON s.proveedor_ruc = b.proveedor_ruc
GROUP BY 1,2""").df()
pa['sup_muy_joven'] = (pa.sup_antiguedad <= 1).astype(float).where(pa.sup_antiguedad.notna())
pa['sup_antes_de_existir'] = (pa.sup_antiguedad < 0).astype(float).where(pa.sup_antiguedad.notna())
print(f'\nAntiguedad al contratar: {pa.sup_antiguedad.describe()[["mean","50%","min","max"]].round(1).to_dict()}')
print(f'Filas con contrato anterior a la constitucion: {int((pa.sup_antiguedad < 0).sum()):,}')

# ---- indicadoras: NULAS para quien no esta en el directorio ----
for c, n_top in [('sup_tipo', 5), ('sup_provincia', 6)]:
    falta = sup[c].isna()
    for v in sup[c].value_counts().head(n_top).index:
        plano = unicodedata.normalize('NFKD', str(v)).encode('ascii','ignore').decode('ascii').lower()
        nom = 'sup_' + re.sub(r'[^a-z0-9]+','_', plano)[:20].strip('_')
        sup[nom] = np.where(falta, np.nan, (sup[c] == v).astype(float))
num_sup = [c for c in sup.columns if c.startswith('sup_') and pd.api.types.is_numeric_dtype(sup[c])]
print(f'\nVariables societarias por empresa ({len(num_sup)}): {num_sup}')
guardar(sup, 'supercias'); guardar(pa, 'supercias_prov_anio')
anotar('3_supercias', n_vars=len(num_sup), cobertura=float(sup.sup_anio_constitucion.notna().mean()))

## Paso 4 — Índice de riesgo tipo CRI, sin Benford

Tres dimensiones con pesos iguales, siguiendo a Fazekas y Kocsis (2020). Cada señal continua se convierte
a percentil **dentro de su año**, y cada dimensión promedia solo las señales disponibles: un faltante no
cuenta como cero ni como uno.

La señal de Benford se retiró porque su estadístico depende del tamaño de muestra y el hallazgo previo
era un artefacto. En su lugar, la dimensión de montos usa el monto frente a la mediana de su CPC y su año,
frente a la mediana de su tipo de procedimiento, y si el monto es sospechosamente redondo.

Este índice es la **línea base** contra la que se mide el modelo.

In [ ]:
# ============ PASO 4 · CRI v2 (sin Benford) ============
cols = set(c[0] for c in con.execute("DESCRIBE bmv").fetchall())
CAND = {
 'competencia': {
   'c_unico':        ('CASE WHEN tiene_periodo_competitivo THEN (CASE WHEN proveedor_unico THEN 1.0 ELSE 0.0 END) END','bin',['tiene_periodo_competitivo','proveedor_unico']),
   'c_conc_rival':   ('concentracion_rival_habitual','pct',['concentracion_rival_habitual']),
   'c_dominancia':   ('dominancia_maxima_historica','pct',['dominancia_maxima_historica']),
   'c_winrate':      ('win_rate','pct',['win_rate'])},
 'capacidad': {
   'k_ratio_capital':('ratio_monto_capital','pct',['ratio_monto_capital']),
   'k_nuevo':        ('CASE WHEN proveedor_es_nuevo_en_mercado THEN 1.0 ELSE 0.0 END','bin',['proveedor_es_nuevo_en_mercado']),
   'k_cap_minimo':   ('CASE WHEN proveedor_capital_suscrito IS NULL THEN NULL WHEN proveedor_capital_suscrito <= 800 THEN 1.0 ELSE 0.0 END','bin',['proveedor_capital_suscrito']),
   'k_joven':        ('-1.0 * anios_desde_primera_actividad_general','pct',['anios_desde_primera_actividad_general'])},
 'montos': {
   'm_vs_cpc':       ('CASE WHEN med_cpc > 0 THEN monto_final / med_cpc END','pct',['cpc5_principal','monto_final']),
   'm_vs_tipo':      ('CASE WHEN med_tipo > 0 THEN monto_final / med_tipo END','pct',['tipo_contratacion','monto_final']),
   'm_redondo':      ('CASE WHEN monto_final IS NULL OR monto_final <= 0 THEN NULL WHEN ABS(monto_final - ROUND(monto_final/1000.0)*1000.0) < 0.005 THEN 1.0 ELSE 0.0 END','bin',['monto_final'])},
}
usar = {}
print('--- Senales ---')
for dim, d in CAND.items():
    usar[dim] = {n:(e,t) for n,(e,t,req) in d.items() if all(c in cols for c in req)}
    falt = [n for n,(e,t,req) in d.items() if not all(c in cols for c in req)]
    print(f'  {dim:12s} usa {list(usar[dim])}' + (f' | FALTAN {falt}' if falt else ''))
    assert usar[dim], f'{dim} se quedo sin senales'

pctl = lambda e: (f'CASE WHEN ({e}) IS NULL THEN NULL ELSE '
                  f'(RANK() OVER (PARTITION BY anio_proceso ORDER BY ({e})) - 1.0) / '
                  f'NULLIF(COUNT({e}) OVER (PARTITION BY anio_proceso) - 1.0, 0) END')
def media_disp(cs):
    num = ' + '.join(f'COALESCE({c},0)' for c in cs)
    den = ' + '.join(f'CASE WHEN {c} IS NULL THEN 0 ELSE 1 END' for c in cs)
    return f'CASE WHEN ({den}) = 0 THEN NULL ELSE ({num}) / ({den}) END'

sel = [f'{pctl(e) if t=="pct" else f"({e})"} AS {n}' for d in usar.values() for n,(e,t) in d.items()]
dims = {dim: media_disp(list(d)) for dim, d in usar.items()}
med_cpc  = 'MEDIAN(monto_final) OVER (PARTITION BY cpc5_principal, anio_proceso)'  if 'cpc5_principal' in cols else 'NULL'
med_tipo = 'MEDIAN(monto_final) OVER (PARTITION BY tipo_contratacion, anio_proceso)' if 'tipo_contratacion' in cols else 'NULL'

t0 = time.time()
con.execute(f"""
COPY (
  WITH b AS (SELECT ocid, proveedor_ruc, anio_proceso, monto_final,
                    {med_cpc} AS med_cpc, {med_tipo} AS med_tipo,
                    * EXCLUDE (ocid, proveedor_ruc, anio_proceso, monto_final) FROM bmv),
       s AS (SELECT ocid, proveedor_ruc, anio_proceso, {', '.join(sel)} FROM b),
       d AS (SELECT *, {', '.join(f'{v} AS dim_{k}' for k, v in dims.items())} FROM s)
  SELECT *, {media_disp([f'dim_{k}' for k in dims])} AS cri2 FROM d
) TO '{OUT}cri_proceso.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
os.replace(f'{OUT}cri_proceso.tmp.parquet', f'{OUT}cri_proceso.parquet')
con.execute(f"CREATE OR REPLACE VIEW cri2 AS SELECT * FROM read_parquet('{OUT}cri_proceso.parquet')")
print(f'\nCRI calculado en {time.time()-t0:.0f} s | {con.execute("SELECT COUNT(*) FROM cri2").fetchone()[0]:,} filas')

print('\nCobertura y correlacion entre dimensiones (deben ser bajas: miden cosas distintas):')
print(con.execute("""SELECT ROUND(100.0*AVG(CASE WHEN dim_competencia IS NOT NULL THEN 1 ELSE 0 END),1) AS cob_competencia,
                            ROUND(100.0*AVG(CASE WHEN cri2 IS NOT NULL THEN 1 ELSE 0 END),1) AS cob_cri,
                            ROUND(CORR(dim_competencia,dim_capacidad),3) AS comp_cap,
                            ROUND(CORR(dim_competencia,dim_montos),3)   AS comp_mon,
                            ROUND(CORR(dim_capacidad,dim_montos),3)     AS cap_mon FROM cri2""").df().to_string(index=False))

con.execute(f"""COPY (
  SELECT proveedor_ruc, anio_proceso, AVG(cri2) AS cri2_medio, MAX(cri2) AS cri2_max,
         AVG(dim_competencia) AS dim_competencia, AVG(dim_capacidad) AS dim_capacidad,
         AVG(dim_montos) AS dim_montos
  FROM cri2 GROUP BY 1,2) TO '{OUT}cri_prov_anio.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
os.replace(f'{OUT}cri_prov_anio.tmp.parquet', f'{OUT}cri_prov_anio.parquet')
n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT}cri_prov_anio.parquet')").fetchone()[0]
print(f'  -> {OUT}cri_prov_anio.parquet ({n:,} filas)')
anotar('4_cri', senales={k: list(v) for k, v in usar.items()}, filas_prov_anio=int(n))

## Paso 5 — Panel de modelado

Una fila por proveedor y año. Las columnas que ya son constantes dentro del proveedor-año (red, co-participación,
comunidad, representante, índices) se toman tal cual; las que varían entre procesos (montos, banderas de
competencia, variables de la entidad compradora) se promedian.

**Dos etiquetas independientes**, como se decidió:

- **A, empresa fantasma**: 15 RUC del catastro del SRI, con fecha.
- **B, competencia simulada**: el proveedor ganó al menos un proceso en el que otra empresa de su mismo
  representante legal también ofertó. 74 RUC, y cero solapamiento con A.

In [ ]:
# ============ PASO 5 · Panel de modelado ============
PREF_CTE = ('red_','cob_','com_','rep_','idx_','grupo_','n_comp_')   # constantes dentro del proveedor-anio
desc = con.execute("DESCRIBE bmv").fetchall()
num = [c for c,t,*_ in desc if any(k in t.upper() for k in ('INT','DOUBLE','FLOAT','DECIMAL','BIGINT'))]
boo = [c for c,t,*_ in desc if 'BOOLEAN' in t.upper()]
llaves = {'anio_proceso','ocid','proveedor_ruc','entidad_ruc'}
cte = [c for c in num+boo if c.startswith(PREF_CTE) and c not in llaves]
var = [c for c in num+boo if not c.startswith(PREF_CTE) and c not in llaves and not c.endswith('_pct')]
print(f'Constantes por proveedor-anio: {len(cte)} | varian por proceso: {len(var)} (ent_* se promedian)')

# ---- etiqueta B: competencia intragrupo contemporanea ----
sup = pd.read_parquet(f'{OUT}supercias.parquet')
gr = sup[['proveedor_ruc','rep_norm']].dropna()
tam = gr.groupby('rep_norm')['proveedor_ruc'].size()
gr = gr[gr.rep_norm.map(tam).between(2, 30)]          # >30 se asume representante nominal
of = con.execute("""SELECT DISTINCT ocid, oferente_ruc AS ruc FROM ofe
                    WHERE TRIM(COALESCE(oferente_ruc,'')) <> ''""").df()
t = of.groupby('ocid')['ruc'].transform('size'); of = of[(t>=2)&(t<=40)]
og = of.merge(gr, left_on='ruc', right_on='proveedor_ruc', how='inner')
cnt = og.groupby(['ocid','rep_norm'])['ruc'].nunique()
casos = cnt[cnt>=2].reset_index()[['ocid','rep_norm']]
print(f'Procesos con 2+ empresas del mismo representante compitiendo: {casos.ocid.nunique():,}')
con.register('casos_t', casos); con.register('gr_t', gr)

agg = (['COUNT(*) AS n_procesos','COUNT(DISTINCT entidad_ruc) AS n_entidades_anio',
        'SUM(monto_final) AS monto_anio','MAX(monto_final) AS monto_max_anio']
       + [f'AVG(CAST("{c}" AS DOUBLE)) AS "{c}"' for c in var]
       + [f'MAX("{c}") AS "{c}"' for c in cte])
t0 = time.time()
con.execute(f"""
COPY (
  WITH pa AS (SELECT proveedor_ruc, anio_proceso, {', '.join(agg)} FROM bmv GROUP BY 1,2),
  cri AS (SELECT * FROM read_parquet('{OUT}cri_prov_anio.parquet')),
  sup AS (SELECT * EXCLUDE (sup_tipo, sup_provincia, rep_norm) FROM read_parquet('{OUT}supercias.parquet')),
  spa AS (SELECT * FROM read_parquet('{OUT}supercias_prov_anio.parquet')),
  ya  AS (SELECT proveedor_ruc, anio_proceso,
                 MAX(CASE WHEN y_fantasma THEN 1 ELSE 0 END)  AS y_fantasma,
                 MAX(CASE WHEN en_catastro THEN 1 ELSE 0 END) AS en_catastro,
                 AVG(CASE WHEN momento = 'antes de la marca' THEN 1.0 ELSE 0.0 END) AS frac_antes_marca
          FROM read_parquet('{OUT}etiqueta.parquet') GROUP BY 1,2),
  ofa AS (SELECT DISTINCT oferente_ruc AS proveedor_ruc, anio AS anio_proceso
          FROM ofe WHERE TRIM(COALESCE(oferente_ruc,'')) <> ''),
  yb  AS (SELECT b.proveedor_ruc, b.anio_proceso,
                 MAX(CASE WHEN c.ocid IS NOT NULL THEN 1 ELSE 0 END) AS y_intragrupo,
                 SUM(CASE WHEN c.ocid IS NOT NULL THEN 1 ELSE 0 END) AS n_proc_intragrupo
          FROM bmv b JOIN gr_t g ON g.proveedor_ruc = b.proveedor_ruc
          LEFT JOIN casos_t c ON c.ocid = b.ocid AND c.rep_norm = g.rep_norm GROUP BY 1,2)
  SELECT pa.*, cri.* EXCLUDE (proveedor_ruc, anio_proceso),
         sup.* EXCLUDE (proveedor_ruc), spa.* EXCLUDE (proveedor_ruc, anio_proceso),
         COALESCE(ya.y_fantasma,0) AS y_fantasma, COALESCE(ya.en_catastro,0) AS en_catastro,
         ya.frac_antes_marca,
         COALESCE(yb.y_intragrupo,0) AS y_intragrupo,
         COALESCE(yb.n_proc_intragrupo,0) AS n_proc_intragrupo,
         (yb.proveedor_ruc IS NOT NULL) AS tiene_grupo,
         (ofa.proveedor_ruc IS NOT NULL) AS aparece_oferentes
  FROM pa
  LEFT JOIN cri ON cri.proveedor_ruc=pa.proveedor_ruc AND cri.anio_proceso=pa.anio_proceso
  LEFT JOIN sup ON sup.proveedor_ruc=pa.proveedor_ruc
  LEFT JOIN spa ON spa.proveedor_ruc=pa.proveedor_ruc AND spa.anio_proceso=pa.anio_proceso
  LEFT JOIN ya  ON ya.proveedor_ruc =pa.proveedor_ruc AND ya.anio_proceso =pa.anio_proceso
  LEFT JOIN yb  ON yb.proveedor_ruc =pa.proveedor_ruc AND yb.anio_proceso =pa.anio_proceso
  LEFT JOIN ofa ON ofa.proveedor_ruc=pa.proveedor_ruc AND ofa.anio_proceso=pa.anio_proceso
) TO '{OUT}panel.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)""")
os.replace(f'{OUT}panel.tmp.parquet', f'{OUT}panel.parquet')
print(f'Panel construido en {time.time()-t0:.0f} s')
con.execute(f"CREATE OR REPLACE VIEW panel AS SELECT * FROM read_parquet('{OUT}panel.parquet')")
n, k = con.execute("SELECT COUNT(*) FROM panel").fetchone()[0], len(con.execute("DESCRIBE panel").fetchall())
print(f'  -> {OUT}panel.parquet ({n:,} filas x {k} columnas)')

print('\n--- Las dos etiquetas ---')
print(con.execute("""SELECT 'A fantasma' AS objetivo, COUNT(*) AS filas_pos,
                            COUNT(DISTINCT proveedor_ruc) AS ruc FROM panel WHERE y_fantasma=1
                     UNION ALL
                     SELECT 'B intragrupo', COUNT(*), COUNT(DISTINCT proveedor_ruc)
                     FROM panel WHERE y_intragrupo=1""").df().to_string(index=False))
print('\nSolapamiento:')
print(con.execute("""SELECT y_fantasma, y_intragrupo, COUNT(*) AS filas FROM panel
                     GROUP BY 1,2 ORDER BY 1 DESC, 2 DESC""").df().to_string(index=False))
print(con.execute("""SELECT
   SUM(CASE WHEN tiene_grupo THEN 1 ELSE 0 END) AS con_grupo,
   SUM(CASE WHEN tiene_grupo AND aparece_oferentes THEN 1 ELSE 0 END) AS universo_B,
   SUM(CASE WHEN y_intragrupo=1 AND NOT aparece_oferentes THEN 1 ELSE 0 END) AS positivos_fuera_del_universo
 FROM panel""").df().to_string(index=False))
print('El universo del objetivo B es quien tiene grupo Y aparece en la tabla de oferentes:')
print('solo esa poblacion pudo, en principio, haber competido contra una empresa hermana.')
anotar('5_panel', filas=int(n), columnas=int(k))

## Paso 6 — Auditoría de fuga y conjuntos de variables

**La fuga se decide por el origen de la variable, no por su AUC.** Una variable no es fuga porque prediga
bien; es fuga porque se derivó de la misma fuente que la etiqueta. Por eso la exclusión es por patrón de
nombre y el AUC solo sirve de aviso para revisión humana.

Dos trampas que ya nos costaron una corrida:

- `proveedor_ruc_estado_riesgoso` y `proveedor_fue_suspendido_alguna_vez` salen del SRI, que es de donde
  sale la etiqueta A. AUC de 0,90. Fuera.
- El AUC del objetivo B debe calcularse **dentro de su universo** (proveedores con grupo identificable).
  Medido sobre todo el panel, `sup_anonima` daba 0,83 cuando dentro del universo real ronda 0,51.

Cuatro conjuntos: completo, anticipado (solo variables de años anteriores, para la pregunta de alerta
temprana) y la variante estricta de cada uno, para reportar sensibilidad.

In [ ]:
# ============ PASO 6 · Auditoria de fuga y conjuntos de variables ============
AVISO_AUC = 0.80
P = pd.read_parquet(f'{OUT}panel.parquet')
print(f'Panel: {len(P):,} filas x {P.shape[1]} columnas')

LLAVES    = {'proveedor_ruc','anio_proceso'}
ETIQUETAS = {'y_fantasma','y_intragrupo','en_catastro','n_proc_intragrupo','tiene_grupo',
             'frac_antes_marca','aparece_oferentes'}
IDS       = {'com_id'}
PAT_SRI   = ('fantasma','estado_riesgoso','suspendid','ruc_estado','catastro','disolucion','liquidac','doc_')
PAT_DIR   = ('sociedad_registrada','en_directorio','tiene_representante')
PAT_B     = ('intragrupo',)
# monto_tender esta nulo en las 216.469 subastas inversas y presente en el 100 % de las otras
# cuatro clases de procedimiento: su patron de nulos ES el tipo de procedimiento, no conducta.
# El monto canonico del trabajo es monto_final, de donde ya sale monto_anio.
PAT_PROC  = ('monto_tender',)
EXC_SRI  = {c for c in P.columns if any(p in c.lower() for p in PAT_SRI)} - ETIQUETAS
EXC_DIR  = {c for c in P.columns if any(p in c.lower() for p in PAT_DIR)}
EXC_PROC = {c for c in P.columns if any(p in c.lower() for p in PAT_PROC)}

num = P.select_dtypes(include=[np.number, bool]).columns
X_cols = [c for c in num if c not in LLAVES | ETIQUETAS | IDS | EXC_SRI | EXC_PROC]
EXC_B_PAT = {c for c in X_cols if any(p in c.lower() for p in PAT_B)}
print('\n--- Exclusiones por origen ---')
print(f'  identificador:                 {sorted(IDS)}')
print(f'  vienen del SRI (A y B):        {sorted(EXC_SRI)}')
print(f'  definicion del grupo (B):      {sorted(EXC_DIR)}')
print(f'  fenomeno rezagado (B):         {sorted(EXC_B_PAT)}')
print(f'  confunde con el procedimiento: {sorted(EXC_PROC)}')
for c in sorted(EXC_PROC):            # queda registrado con su cifra, para poder defenderlo
    print(f'     {c}: AUC_A {auc(P[c].values, P.y_fantasma.values):.3f} | '
          f'nulos {pd.isna(P[c]).mean()*100:.1f} %')
print(f'\nCandidatas: {len(X_cols)}')

univ_B = (P['tiene_grupo'] & P['aparece_oferentes']).values.astype(bool)   # poblacion en riesgo de B
aud = pd.DataFrame([(c, auc(P[c].values, P.y_fantasma.values),
                        auc(P[c].values[univ_B], P.y_intragrupo.values[univ_B]),
                        float(pd.isna(P[c]).mean())) for c in X_cols],
                   columns=['variable','auc_A','auc_B','pct_nulos'])
lim = AVISO_AUC - 0.5
avisa = aud[(aud.auc_A.sub(0.5).abs() > lim) | (aud.auc_B.sub(0.5).abs() > lim)]
print(f'\n--- AVISO: |AUC-0.5| > {lim:.2f} (revisar el origen de cada una) ---')
print(avisa.round(3).to_string(index=False) if len(avisa) else '  ninguna')
if len(avisa):
    print('Una AUC alta no es motivo para excluir: la exclusion se decide por ORIGEN. Si nace del')
    print('SRI, del directorio o del procedimiento, agregala al patron y vuelve a correr.')
    print('Las rep_*intragrupo* aparecen aqui pero NO entran al objetivo B: exc_B las saca.')

FAM = lambda pre: [c for c in X_cols if c.startswith(pre)]
fam = {'red':FAM('red_'), 'cob':FAM('cob_'), 'com':FAM('com_'),
       'rep':[c for c in X_cols if c.startswith('rep_') or c.startswith('grupo_')],
       'ent':FAM('ent_'), 'idx':FAM('idx_'), 'sup':FAM('sup_'),
       'cri':[c for c in X_cols if c.startswith('cri2') or c.startswith('dim_')]}
fam['base'] = [c for c in X_cols if c not in sum(fam.values(), [])]

exc_B = set(fam['rep']) | EXC_DIR | EXC_B_PAT
ANT   = fam['red'] + fam['cob'] + fam['com'] + fam['idx']     # solo informacion de anios anteriores
alto  = set(avisa.variable)
VARS = {'A': X_cols,                      'B': [c for c in X_cols if c not in exc_B],
        'A_anticipada': ANT,              'B_anticipada': [c for c in ANT if c not in exc_B],
        'A_estricta': [c for c in X_cols if c not in alto],
        'B_estricta': [c for c in X_cols if c not in exc_B and c not in alto]}
print('\n--- Conjuntos ---')
for k, v in VARS.items(): print(f'  {k:14s} {len(v):3d} variables')
print('\n--- Familias ---')
for f, cs in fam.items():
    if cs: print(f'  {f:5s} {len(cs):3d} vars | nulos promedio {P[cs].isna().mean().mean()*100:5.1f}%')

for obj, c_auc in [('A','auc_A'), ('B','auc_B')]:
    s = aud[aud.variable.isin(VARS[obj])]
    print(f'\n--- Mas informativas del objetivo {obj} ---')
    print(s.reindex(s[c_auc].sub(0.5).abs().sort_values(ascending=False).index).head(8).round(3).to_string(index=False))

with open(f'{OUT}vars.json','w',encoding='utf-8') as f: json.dump(VARS, f, ensure_ascii=False, indent=1)
with open(f'{OUT}familias.json','w',encoding='utf-8') as f: json.dump(fam, f, ensure_ascii=False, indent=1)
aud.to_csv(f'{OUT}auditoria.csv', index=False)
print(f'\n  -> {OUT}vars.json, {OUT}familias.json, {OUT}auditoria.csv')
print('OJO: esta celda REESCRIBE vars.json, asi que borra A_sin_ausencia y B_sin_ausencia.')
print('Los vuelve a crear el Paso 14. El orden es 6 -> 7b -> 12 -> 14.')
anotar('6_auditoria', candidatas=len(X_cols), excluidas_sri=sorted(EXC_SRI),
       excluidas_procedimiento=sorted(EXC_PROC), avisos=sorted(alto))

## Paso 7 — Modelo PU y evaluación

**Bagging PU** (Mordelet y Vert, 2014): cada modelo del conjunto ve todos los positivos y una muestra de
no etiquetados; el puntaje final es el promedio del conjunto.

**La evaluación es lo que da validez al resultado.** Para el objetivo A, con 15 empresas, se deja una
empresa fuera por vez. Para el objetivo B, con 74, validación cruzada en cinco pliegues agrupados **por
representante legal**, no por empresa: dos empresas del mismo dueño son el mismo caso.

La métrica es el percentil del positivo dentro de su año. La hipótesis nula se construye sorteando
conglomerados al azar del mismo tamaño, así el valor p responde la pregunta correcta: si hubiera señalado
al azar la misma cantidad de empresas, ¿con qué frecuencia habría obtenido un orden así de bueno?

Se compara siempre contra dos líneas base: el CRI v2 y el índice de intermediación.

In [ ]:
# ============ PASO 7b · Evaluacion reproducible (reemplaza al Paso 7 para los numeros finales) ============
# Por que hace falta. Entre la corrida anterior y esta, los casos "anticipada" tenian EXACTAMENTE
# el mismo conjunto de 46 variables y los mismos datos, y aun asi se movieron:
#   A anticipada  0,673 -> 0,422     B anticipada  0,717 -> 0,516
# Los indices fijos, que no se entrenan, dieron casi identico (0,870 vs 0,868; 0,708 vs 0,708).
# O sea: los datos son los mismos y lo que se movio fue el modelo. La causa es que evaluar() usa
# el `rng` GLOBAL para la muestra de comparacion y para la particion de pliegues, asi que el
# resultado depende de cuantas celdas consumieron azar antes. Eso no es aceptable para una cifra
# que va a un documento.
# Esta celda usa semilla explicita y promedia sobre varias semillas, de modo que lo que se reporta
# es una media con su recorrido, no un sorteo. Usa la MISMA configuracion que el Paso 12, para que
# el valor observado sea directamente comparable contra el nulo de permutacion.
try: from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    import subprocess; subprocess.run([sys.executable,'-m','pip','install','scikit-learn'], check=True)
    from sklearn.ensemble import HistGradientBoostingClassifier

K_EV, R_SEM, N_MUESTRA, SEM_MUESTRA = 25, 5, 600, 777

P = pd.read_parquet(f'{OUT}panel.parquet')
P = P.merge(pd.read_parquet(f'{OUT}supercias.parquet')[['proveedor_ruc','rep_norm']],
            on='proveedor_ruc', how='left')
P['grupo_B'] = P.rep_norm.fillna('ruc:' + P.proveedor_ruc)
with open(f'{OUT}vars.json', encoding='utf-8') as f: VARS = json.load(f)
print('Conjuntos cargados:', {k: len(v) for k, v in VARS.items()})
if 'monto_tender' in VARS['A']:
    print('\n  AVISO: monto_tender sigue en el conjunto A. Aplica el parche al Paso 6 y vuelve')
    print('  a correr el Paso 6 antes de seguir, o esta celda medira el modelo equivocado.\n')

def muestra_comparacion(y, anios, semilla=SEM_MUESTRA, n_por_anio=N_MUESTRA):
    r = np.random.default_rng(semilla); partes = []
    for a in np.unique(anios):
        m = np.where((y == 0) & (anios == a))[0]
        if len(m): partes.append(r.choice(m, size=min(n_por_anio, len(m)), replace=False))
    return np.concatenate(partes)

def pu_bag(Xp, Xu, Xs, k, seed, ratio=5):
    r = np.random.default_rng(seed); acum = np.zeros(len(Xs)); n = len(Xp); hechos = 0
    with contextlib.redirect_stderr(io.StringIO()):
        for i in range(k):
            idx = r.choice(len(Xu), size=min(len(Xu), n*ratio), replace=False)
            Xtr = np.vstack([Xp, Xu[idx]]); ytr = np.r_[np.ones(n), np.zeros(len(idx))]
            util = np.array([np.unique(Xtr[~np.isnan(Xtr[:,j]), j]).size >= 2 for j in range(Xtr.shape[1])])
            if util.sum() == 0: continue
            m = HistGradientBoostingClassifier(max_depth=3, max_iter=60, learning_rate=0.1,
                    min_samples_leaf=5, l2_regularization=1.0, random_state=i).fit(Xtr[:,util], ytr)
            acum += m.predict_proba(Xs[:,util])[:,1]; hechos += 1
    assert hechos > 0, 'Ningun modelo se pudo entrenar'
    return acum / hechos

def estadistico(X, y, gi, anios, n_folds, k, semilla, Xu=None):
    """Percentil mediano de los positivos dentro de su anio. Todo el azar sale de `semilla`:
    nada depende de que celdas se corrieron antes. La poblacion de comparacion la puntua el
    promedio de todos los pliegues, no uno solo."""
    r = np.random.default_rng(semilla); gp = np.unique(gi[y == 1])
    if len(gp) < 2: return np.nan
    idx_s = np.concatenate([np.where(y == 1)[0], muestra_comparacion(y, anios)])
    Xs = X[idx_s]
    if Xu is None: Xu = X[y == 0]
    score = np.full(len(idx_s), np.nan); acum = np.zeros(len(idx_s)); nf = 0
    folds = ([[c] for c in gp] if n_folds is None
             else [f for f in np.array_split(r.permutation(gp), min(n_folds, len(gp))) if len(f)])
    for j, fuera in enumerate(folds):
        dentro = np.isin(gi, fuera)
        s = pu_bag(X[(y == 1) & ~dentro], Xu, Xs, k, semilla + j)
        donde = dentro[idx_s] & (y[idx_s] == 1)
        score[donde] = s[donde]; acum += s; nf += 1
    falta = np.isnan(score); score[falta] = (acum / nf)[falta]
    ev = pd.DataFrame({'y': y[idx_s], 'anio': anios[idx_s], 'm': score})
    pct = ev.groupby('anio')['m'].rank(pct=True).values
    return float(np.nanmedian(pct[ev.y.values == 1]))

def percentil_indice(d, col, y):
    if col not in d.columns: return np.nan
    pz = d.groupby('anio_proceso')[col].rank(pct=True).values
    return float(np.nanmedian(pz[y == 1]))

uA = P.aparece_oferentes.values.astype(bool)
uB = (P.tiene_grupo & P.aparece_oferentes).values.astype(bool)
CASOS = [
 ('A fantasma · completo',           P,     'y_fantasma',   'proveedor_ruc', 'A',            None),
 ('A fantasma · universo oferentes', P[uA], 'y_fantasma',   'proveedor_ruc', 'A',            None),
 ('A fantasma · anticipada',         P,     'y_fantasma',   'proveedor_ruc', 'A_anticipada', None),
 ('B intragrupo · completo',         P[uB], 'y_intragrupo', 'grupo_B',       'B',            5),
 ('B intragrupo · anticipada',       P[uB], 'y_intragrupo', 'grupo_B',       'B_anticipada', 5),
]
filas, t0 = [], time.time()
for nombre, d, cy, cg, conj, nf in CASOS:
    cols = VARS[conj]
    y  = d[cy].values.astype(int); gi = pd.factorize(d[cg].values)[0]
    an = d.anio_proceso.values;    X  = d[cols].astype(float).values
    Xu = X[y == 0]
    vals = [estadistico(X, y, gi, an, nf, K_EV, SEMILLA + 31*s, Xu=Xu) for s in range(R_SEM)]
    v = np.array([x for x in vals if np.isfinite(x)])
    filas.append({'caso': nombre, 'metodo': 'modelo', 'n_vars': len(cols),
                  'media': round(v.mean(), 3), 'min': round(v.min(), 3), 'max': round(v.max(), 3),
                  'recorrido': round(v.max() - v.min(), 3), 'n_semillas': len(v)})
    for nom, c in [('cri_v2', 'cri2_medio'), ('idx_intermediacion', 'idx_intermediacion')]:
        p_ = percentil_indice(d, c, y)
        if np.isfinite(p_):
            filas.append({'caso': nombre, 'metodo': nom, 'n_vars': 1, 'media': round(p_, 3),
                          'min': round(p_, 3), 'max': round(p_, 3), 'recorrido': 0.0, 'n_semillas': 1})
    print(f'  {nombre:34s} modelo {v.mean():.3f} [{v.min():.3f}-{v.max():.3f}]  [{time.time()-t0:.0f} s]')

res = pd.DataFrame(filas)
print('\n' + '='*78 + '\nRESULTADOS REPRODUCIBLES\n' + '='*78)
print(res.to_string(index=False))
res.to_csv(f'{OUT}resultados_v2.csv', index=False)
print(f'\n  -> {OUT}resultados_v2.csv')
print('\nAqui NO hay valor p: el nulo correcto es el de permutacion del Paso 12, y la columna')
print('`media` de esta tabla es el valor observado que hay que comparar contra ese nulo.')
print('El recorrido entre semillas es la incertidumbre del modelo, y va declarado en la tesis.')
anotar('7b_reproducible', K_EV=K_EV, R_SEM=R_SEM, tabla=res.to_dict('records'))

## Paso 8 — Ablación por familia de variables

El paso anterior dice *cuánto* acierta el modelo. Este dice *gracias a qué*. Se reentrena quitando una
familia completa por vez y se mide cuánto cae el desempeño: la caída es el aporte de esa familia.

Es la respuesta directa a la pregunta de la tesis, ¿las variables de red aportan por encima de un índice
de riesgo convencional?, y es más honesta que una medida de importancia calculada sobre el modelo ya
ajustado, que solo dice en qué se apoyó y no si era necesario.

Se corre sobre el objetivo B, que es el que tiene potencia estadística: 74 empresas contra 15.

In [ ]:
# ============ PASO 8 · Ablacion por familia ============
K_ABL = 40          # menos modelos por conjunto: son muchas corridas y solo se comparan entre si
fam = json.load(open(f'{OUT}familias.json', encoding='utf-8'))
base_cols = VARS['B']
familias = {f: [c for c in cs if c in base_cols] for f, cs in fam.items()}
familias = {f: cs for f, cs in familias.items() if cs}
print('Familias presentes en el objetivo B:', {f: len(cs) for f, cs in familias.items()})

_k = K_BAG; K_BAG = K_ABL      # temporal, para que pu_bagging use menos modelos
filas = []
t0 = time.time()
VARS['_ablacion_completo'] = base_cols
r0, _ = evaluar('B', '_ablacion_completo', n_folds=5, etiqueta='B completo (referencia)', verbose=False)
ref = float(r0.loc[r0.metodo == 'modelo', 'mediana_percentil'].iloc[0])
ref_lift = float(r0.loc[r0.metodo == 'modelo', 'lift_decil'].iloc[0])
filas.append(('(ninguna)', len(base_cols), ref, ref_lift, 0.0))
print(f'Referencia con las {len(base_cols)} variables: mediana {ref:.3f} | lift {ref_lift:.1f}')

for f, cs in familias.items():
    VARS['_abl'] = [c for c in base_cols if c not in cs]
    if len(VARS['_abl']) < 5: print(f'  {f}: quedarian {len(VARS["_abl"])} variables, se omite'); continue
    r, _ = evaluar('B', '_abl', n_folds=5, etiqueta=f'sin {f}', verbose=False)
    m = float(r.loc[r.metodo=='modelo','mediana_percentil'].iloc[0])
    l = float(r.loc[r.metodo=='modelo','lift_decil'].iloc[0])
    filas.append((f'sin {f}', len(VARS['_abl']), m, l, round(ref - m, 3)))
    print(f'  sin {f:5s} ({len(cs):3d} vars fuera) -> mediana {m:.3f} | lift {l:4.1f} | caida {ref-m:+.3f}')
K_BAG = _k
abl = pd.DataFrame(filas, columns=['quita','n_vars','mediana_percentil','lift_decil','caida'])
abl = abl.sort_values('caida', ascending=False)
abl.to_csv(f'{OUT}ablacion.csv', index=False)
print(f'\n--- Aporte de cada familia (caida mayor = mas necesaria) --- [{time.time()-t0:.0f} s]')
print(abl.to_string(index=False))
print(f'\n  -> {OUT}ablacion.csv')
anotar('8_ablacion', referencia=ref, filas=abl.to_dict('records'))

## Paso 9 — Alerta temprana

La pregunta operativa no es si el modelo reconoce a una empresa ya marcada, sino si la habría señalado
**antes** de que el SRI la marcara. De las filas de empresas del catastro, la mayoría corresponde a
contratos anteriores a la marca.

No hace falta reentrenar: en la validación dejando una empresa fuera, **todas** las filas de la empresa
evaluada son fuera de muestra, incluidas las anteriores a la marca. Basta recalcular la métrica sobre
ese subconjunto de positivos.

In [ ]:
# ============ PASO 9 · Alerta temprana ============
etq = pd.read_parquet(f'{OUT}etiqueta.parquet')
antes = (etq[etq.en_catastro]
         .assign(es_antes=lambda d: (d.momento == 'antes de la marca').astype(float))
         .groupby(['proveedor_ruc','anio_proceso'])['es_antes'].mean()
         .rename('frac_antes').reset_index())
ev = evA.merge(antes, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_proceso'], how='left')
ev['antes_marca'] = (ev.frac_antes.fillna(0) > 0.5) & (ev.y == 1)
print(f'Positivos evaluados: {int((ev.y==1).sum())} | de ellos anteriores a la marca: {int(ev.antes_marca.sum())}'
      f' en {ev.loc[ev.antes_marca,"proveedor_ruc"].nunique()} empresas')

filas = []
for met in [c for c in ['modelo','cri_v2','idx_intermediacion'] if c in ev]:
    pct = ev.groupby('anio')[met].rank(pct=True).values
    for sub, nom in [(ev.y.values == 1, 'todos los positivos'),
                     (ev.antes_marca.values, 'solo anteriores a la marca')]:
        if sub.sum() < 5: continue
        obs = float(np.nanmedian(pct[sub]))
        tam = ev.loc[sub, 'g'].nunique()
        nul = nulo_por_conglomerado(ev, pct, tam, semilla=SEMILLA)
        p = (1 + np.sum(np.abs(nul-0.5) >= abs(obs-0.5))) / (B_NULL + 1)
        t10 = float(np.nanmean(pct[sub] > 0.9))
        filas.append((met, nom, int(sub.sum()), tam, round(obs,3),
                      round(float(np.percentile(nul,2.5)),3), round(float(np.percentile(nul,97.5)),3),
                      round(p,4), round(t10/0.10,1)))
at = pd.DataFrame(filas, columns=['metodo','subconjunto','n_filas','n_empresas','mediana_percentil',
                                  'nulo_p2.5','nulo_p97.5','p','lift_decil'])
at.to_csv(f'{OUT}alerta_temprana.csv', index=False)
print('\n--- Antes de la marca del SRI ---')
print(at.to_string(index=False))
print('\nSi la fila "solo anteriores a la marca" conserva el valor p, el metodo anticipa y no solo reconoce.')
print(f'\n  -> {OUT}alerta_temprana.csv')
anotar('9_alerta_temprana', filas=at.to_dict('records'))

## Paso 10 — Lista priorizada para revisión

El producto que de verdad sirve a una entidad de control: un listado ordenado por riesgo estimado, con el
motivo de cada señalamiento en lenguaje legible. El modelo se entrena aquí con **todos** los positivos,
porque ya no se trata de evaluar sino de producir.

Advertencia que debe acompañar al archivo: un puntaje alto no es evidencia de infracción. Es una
prioridad de revisión.

In [ ]:
# ============ PASO 10 · Lista priorizada ============
TOP_N, ANIO_DESDE = 300, 2022
P = pd.read_parquet(f'{OUT}panel.parquet')
P = P.merge(pd.read_parquet(f'{OUT}supercias.parquet')[['proveedor_ruc','rep_norm']], on='proveedor_ruc', how='left')

nombres = con.execute("""SELECT proveedor_ruc, ANY_VALUE(proveedor_nombre) AS nombre,
                                SUM(monto_final) AS monto_total, COUNT(DISTINCT ocid) AS procesos
                         FROM bmv GROUP BY 1""").df()

listas = []
for obj, col_y, conjunto in [('A','y_fantasma','A'), ('B','y_intragrupo','B')]:
    d = P[P.tiene_grupo & P.aparece_oferentes] if obj == 'B' else P
    cols = VARS[conjunto]
    X = d[cols].astype(float).values; y = d[col_y].values.astype(int)
    if y.sum() < 2: continue
    print(f'\nObjetivo {obj}: entrenando con los {y.sum()} positivos sobre {len(d):,} filas...')
    t0 = time.time()
    s = pu_bagging(X[y == 1], X[y == 0], X, seed=SEMILLA)
    print(f'  puntuadas {len(d):,} filas en {time.time()-t0:.0f} s')
    r = d[['proveedor_ruc','anio_proceso','n_procesos','monto_anio']].copy()
    r['objetivo'] = obj; r['score'] = s
    r['pct_anio'] = r.groupby('anio_proceso')['score'].rank(pct=True)
    r['ya_etiquetado'] = y
    listas.append(r)
pri = pd.concat(listas, ignore_index=True)

# ---- motivo legible: las 3 variables mas extremas respecto de su año ----
#      las columnas de desvio se prefijan con z_ para no chocar con las del panel
EXPL = {c: n for c, n in [
    ('idx_intermediacion','indice de intermediacion alto'), ('red_autovector','posicion central en la red'),
    ('red_n_entidades','muchas entidades compradoras'),     ('cob_n_rivales','muchos rivales distintos'),
    ('cob_agrupamiento','sin grupo cerrado de rivales'),    ('com_participacion','opera entre varias comunidades'),
    ('ratio_monto_capital','monto muy superior al capital'),('sup_antiguedad','empresa joven al contratar'),
    ('proveedor_unico','gana sin competencia'),             ('monto_anio','monto anual alto'),
    ('red_hhi_entidades','cartera de compradores dispersa'),('sup_frac_fuera_provincia','contrata fuera de su provincia')]
    if c in P.columns}
Z = P[['proveedor_ruc','anio_proceso']].copy()
for c in EXPL:
    g = P.groupby('anio_proceso')[c]
    Z['z_' + c] = ((P[c] - g.transform('mean')) / g.transform('std').replace(0, np.nan))
Z_COLS = ['z_' + c for c in EXPL]
def motivo(fila):
    v = fila[Z_COLS].astype(float).dropna()
    if v.empty: return '(sin variables comparables)'
    v = v.abs().sort_values(ascending=False).head(3)
    return '; '.join(f'{EXPL[c[2:]]} ({fila[c]:+.1f} sigma)' for c in v.index)

pri = (pri.merge(nombres, on='proveedor_ruc', how='left')
          .merge(Z, on=['proveedor_ruc','anio_proceso'], how='left'))
top = (pri[(pri.anio_proceso >= ANIO_DESDE) & (pri.ya_etiquetado == 0)]
         .sort_values('score', ascending=False).head(TOP_N).copy())
top['motivo'] = top.apply(motivo, axis=1)
sal = top[['objetivo','anio_proceso','proveedor_ruc','nombre','n_procesos','monto_anio',
           'score','pct_anio','motivo']].rename(columns={'anio_proceso':'anio','monto_anio':'monto'})
sal.to_csv(f'{OUT}priorizacion.csv', index=False, encoding='utf-8-sig')
pri[['objetivo','proveedor_ruc','anio_proceso','score','pct_anio','ya_etiquetado']].to_parquet(f'{OUT}scores_todos.parquet', index=False)
print(f'\n--- Top 15 de {TOP_N} (desde {ANIO_DESDE}, excluyendo los ya etiquetados) ---')
print(sal.head(15).to_string(index=False, max_colwidth=46))
print(f'\n  -> {OUT}priorizacion.csv ({len(sal)} filas), {OUT}scores_todos.parquet')
print('\nADVERTENCIA para el archivo: un puntaje alto NO es evidencia de infraccion.')
print('Es una prioridad de revision. La verificacion documental la hace la entidad de control.')
anotar('10_priorizacion', top_n=TOP_N, desde=ANIO_DESDE)

# ---- cierre ----
MANIFIESTO['fin'] = time.strftime('%Y-%m-%d %H:%M:%S')
with open(f'{OUT}manifiesto.json','w',encoding='utf-8') as f: json.dump(MANIFIESTO, f, ensure_ascii=False, indent=1)
print('\n================ ARCHIVOS DEL CUADERNO 8 ================')
for f in sorted(glob.glob(f'{OUT}*')):
    print(f'  {os.path.getsize(f)/1e6:8.2f} MB  {os.path.basename(f)}')

## Paso 11 — Pruebas de robustez

Tres pruebas que un revisor exigente va a pedir, y que conviene traer hechas.

**11.1 ¿El modelo aprende comportamiento o aprende quién tiene datos?** La ausencia de información no es
aleatoria: la familia de co-participación tiene 57,8% de nulos y la de representante 94,7%. Si los
positivos tienen un patrón de faltantes distinto al resto, el modelo puede estar detectando *cobertura*
en lugar de *conducta*. La prueba: entrenar un modelo **solo con indicadores de ausencia**, sin ningún
valor. Si ese modelo rinde parecido al completo, el resultado principal no vale.

**11.2 Control negativo.** Se baraja la etiqueta entre empresas, conservando el número de positivos y su
estructura de conglomerados, y se corre la tubería entera. Debe dar mediana cercana a 0,50 y valor p alto.
Si da señal con etiquetas falsas, la señal la fabrica el procedimiento y no los datos. Es la prueba más
convincente que existe contra la sospecha de sobreajuste.

**11.3 Estabilidad.** El mismo cálculo con varias semillas. Si el resultado se mueve mucho, no es un
resultado.

In [ ]:
# ============ PASO 11 · Pruebas de robustez ============
K_ROB, B_ROB = 40, 2000          # mas livianos: son comparaciones entre si, no la cifra final

def evaluar_crudo(X, y, g, anios, rucs, n_folds=None, k=K_ROB, semilla=SEMILLA, B=B_ROB):
    """Version minima de la evaluacion, para las pruebas de robustez. Misma logica del Paso 7."""
    r = np.random.default_rng(semilla)
    gp = pd.unique(g[y == 1])
    if len(gp) < 2: return np.nan, np.nan, np.nan, len(gp)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.concatenate([r.choice(np.where((y==0)&(anios==a))[0],
                     size=min(N_POR_ANIO, ((y==0)&(anios==a)).sum()), replace=False)
                     for a in np.unique(anios) if ((y==0)&(anios==a)).sum() > 0])
    idx_s = np.concatenate([idx_pos, idx_neg]); Xs = X[idx_s]
    score = np.full(len(idx_s), np.nan)
    folds = ([[c] for c in gp] if n_folds is None
             else [f for f in np.array_split(r.permutation(gp), min(n_folds, len(gp))) if len(f)])
    base = np.zeros(len(Xs))
    for j, fuera in enumerate(folds):
        m_tr = (y == 1) & (~pd.Series(g).isin(fuera).values)
        s = pu_bagging(X[m_tr], X[y == 0], Xs, k=k, seed=semilla + j)
        donde = np.isin(g[idx_s], fuera) & (y[idx_s] == 1)
        score[donde] = s[donde]
        if j == 0: base = s
    score[np.isnan(score)] = base[np.isnan(score)]
    ev = pd.DataFrame({'g': g[idx_s], 'y': y[idx_s], 'anio': anios[idx_s], 'm': score})
    pct = ev.groupby('anio')['m'].rank(pct=True).values
    obs = float(np.nanmedian(pct[ev.y.values == 1]))
    nul = nulo_por_conglomerado(ev, pct, len(gp), B=B, semilla=semilla)
    p = (1 + np.sum(np.abs(nul-0.5) >= abs(obs-0.5))) / (B + 1)
    lift = float(np.nanmean(pct[ev.y.values==1] > 0.9)) / 0.10
    return obs, p, lift, len(gp)

P = pd.read_parquet(f'{OUT}panel.parquet')
P = P.merge(pd.read_parquet(f'{OUT}supercias.parquet')[['proveedor_ruc','rep_norm']], on='proveedor_ruc', how='left')
P['grupo_B'] = P.rep_norm.fillna('ruc:' + P.proveedor_ruc)
VARS = json.load(open(f'{OUT}vars.json', encoding='utf-8'))
CONF = {'A': dict(d=P,                 col_y='y_fantasma',   col_g='proveedor_ruc', folds=None, cols=VARS['A']),
        'B': dict(d=P[P.tiene_grupo & P.aparece_oferentes], col_y='y_intragrupo',
                  col_g='grupo_B', folds=5, cols=VARS['B'])}

# ---------- 11.1 ¿Aprende comportamiento o cobertura? ----------
print('='*64 + '\n11.1  Modelo construido SOLO con indicadores de ausencia\n' + '='*64)
rob1 = []
for obj, cf in CONF.items():
    d = cf['d']; y = d[cf['col_y']].values.astype(int)
    g = d[cf['col_g']].values; an = d.anio_proceso.values
    Xv = d[cf['cols']].astype(float).values                      # valores reales
    Xm = np.isnan(Xv).astype(float)                              # SOLO si falta o no
    util = Xm.std(axis=0) > 0
    o1, p1, l1, _ = evaluar_crudo(Xv, y, g, an, d.proveedor_ruc.values, cf['folds'])
    o2, p2, l2, _ = evaluar_crudo(Xm[:, util], y, g, an, d.proveedor_ruc.values, cf['folds'])
    rob1 += [(obj, 'valores reales', round(o1,3), round(p1,4), round(l1,1)),
             (obj, 'solo ausencia',  round(o2,3), round(p2,4), round(l2,1))]
    print(f'  objetivo {obj}: valores {o1:.3f} (p={p1:.4f}, lift {l1:.1f}) | '
          f'solo ausencia {o2:.3f} (p={p2:.4f}, lift {l2:.1f})')
rob1 = pd.DataFrame(rob1, columns=['objetivo','variables','mediana_percentil','p','lift_decil'])
print('\nLectura: si "solo ausencia" se acerca a "valores reales", el modelo detecta cobertura, no conducta.')

print('\n--- Tasa de nulos por familia: positivos frente al resto ---')
fam = json.load(open(f'{OUT}familias.json', encoding='utf-8'))
filas = []
uB = (P.tiene_grupo & P.aparece_oferentes)       # B se compara DENTRO de su universo
for f, cs in fam.items():
    cs = [c for c in cs if c in P.columns]
    if not cs: continue
    na = P[cs].isna().mean(axis=1)
    filas.append((f, len(cs),
                  round(na[P.y_fantasma==1].mean(),3),  round(na[P.y_fantasma==0].mean(),3),
                  round(na[uB & (P.y_intragrupo==1)].mean(),3), round(na[uB & (P.y_intragrupo==0)].mean(),3)))
nf = pd.DataFrame(filas, columns=['familia','n_vars','nulos_A_pos','nulos_A_resto','nulos_B_pos','nulos_B_resto'])
nf['brecha_A'] = (nf.nulos_A_pos - nf.nulos_A_resto).round(3)
nf['brecha_B'] = (nf.nulos_B_pos - nf.nulos_B_resto).round(3)
print(nf.to_string(index=False))
print('Brecha negativa = los positivos tienen MAS datos que el resto, que es el efecto de seleccion')
print('ya conocido: una empresa sobrevive lo suficiente para acumular historial. Una brecha grande')
print('en una familia concreta es la que obliga a revisar esa familia.')

# ---------- 11.2 Control negativo ----------
print('\n' + '='*64 + '\n11.2  Control negativo: etiquetas barajadas entre empresas\n' + '='*64)
N_PLACEBO = 5
rob2 = []
for obj, cf in CONF.items():
    d = cf['d']; y = d[cf['col_y']].values.astype(int)
    g = d[cf['col_g']].values; an = d.anio_proceso.values
    X = d[cf['cols']].astype(float).values
    gp_real = pd.unique(g[y == 1]); gneg = pd.unique(g[~pd.Series(g).isin(gp_real).values])
    if len(gneg) < len(gp_real):
        print(f'  objetivo {obj}: solo {len(gneg)} conglomerados sin positivos para {len(gp_real)} '
              'necesarios. No se puede barajar la etiqueta; se omite el control negativo.')
        continue
    for s in range(N_PLACEBO):
        r = np.random.default_rng(SEMILLA + 1000 + s)
        falsos = r.choice(gneg, size=len(gp_real), replace=False)
        yp = np.isin(g, falsos).astype(int)
        o, p, l, _ = evaluar_crudo(X, yp, g, an, d.proveedor_ruc.values, cf['folds'], semilla=SEMILLA+1000+s)
        rob2.append((obj, s, round(o,3), round(p,4), round(l,1)))
        print(f'  objetivo {obj} · repeticion {s+1}: mediana {o:.3f} | p {p:.4f} | lift {l:.1f}')
rob2 = pd.DataFrame(rob2, columns=['objetivo','repeticion','mediana_percentil','p','lift_decil'])
print('\nEsperado: medianas alrededor de 0,50 y valores p altos. Si alguna da senal, el')
print('procedimiento fabrica resultados y hay que revisarlo antes de seguir.')
print(rob2.groupby('objetivo')[['mediana_percentil','lift_decil']].agg(['mean','min','max']).round(3).to_string())

# ---------- 11.3 Estabilidad entre semillas ----------
print('\n' + '='*64 + '\n11.3  Estabilidad entre semillas\n' + '='*64)
rob3 = []
for obj, cf in CONF.items():
    d = cf['d']; y = d[cf['col_y']].values.astype(int)
    g = d[cf['col_g']].values; an = d.anio_proceso.values
    X = d[cf['cols']].astype(float).values
    for s in [SEMILLA, SEMILLA+7, SEMILLA+13]:
        o, p, l, ng = evaluar_crudo(X, y, g, an, d.proveedor_ruc.values, cf['folds'], semilla=s)
        rob3.append((obj, s, round(o,3), round(p,4), round(l,1)))
        print(f'  objetivo {obj} · semilla {s}: mediana {o:.3f} | p {p:.4f} | lift {l:.1f}')
rob3 = pd.DataFrame(rob3, columns=['objetivo','semilla','mediana_percentil','p','lift_decil'])
print('\nRango entre semillas:')
print(rob3.groupby('objetivo')['mediana_percentil'].agg(['min','max']).assign(
      rango=lambda t: (t['max']-t['min']).round(3)).round(3).to_string())

pd.concat([rob1.assign(prueba='11.1 ausencia'), rob2.assign(prueba='11.2 control negativo'),
           rob3.assign(prueba='11.3 estabilidad')]).to_csv(f'{OUT}robustez.csv', index=False)
nf.to_csv(f'{OUT}nulos_por_familia.csv', index=False)
print(f'\n  -> {OUT}robustez.csv, {OUT}nulos_por_familia.csv')
anotar('11_robustez', solo_ausencia=rob1.to_dict('records'),
       placebo_media={k: float(v) for k, v in rob2.groupby('objetivo')['mediana_percentil'].mean().items()},
       estabilidad={k: float(v) for k, v in rob3.groupby('objetivo')['mediana_percentil'].std().items()})

In [ ]:
# ============ PASO 12 · Prueba de permutacion de etiquetas ============
# Por que hace falta. El nulo del Paso 7 sortea conglomerados al azar y los puntua con el modelo
# YA entrenado con los positivos reales. Para un metodo ENTRENADO ese nulo no es el correcto, y se
# ve en el control negativo del 11.2: tres de cinco placebos del objetivo A salieron "significativos"
# por debajo de 0,50. Aqui el nulo se construye repitiendo TODO el procedimiento con las etiquetas
# permutadas entre conglomerados emparejados por numero de filas. Este es el valor p que se reporta.

N_PERM    = 49      # 49 da resolucion de 0,02. Subir a 99 si sobra tiempo (el doble de duracion).
K_P       = 15      # bolsas por pliegue. EL MISMO para lo observado y para el nulo: si el estadistico
                    # no se calcula igual en los dos lados, la comparacion no vale nada.
N_MUESTRA = 600     # no etiquetados por anio en la poblacion de comparacion (basta para un percentil)
SEM_MUESTRA = 777   # la muestra de comparacion se fija: entre permutaciones solo cambia la etiqueta

P = pd.read_parquet(f'{OUT}panel.parquet')
P = P.merge(pd.read_parquet(f'{OUT}supercias.parquet')[['proveedor_ruc','rep_norm']], on='proveedor_ruc', how='left')
P['grupo_B'] = P.rep_norm.fillna('ruc:' + P.proveedor_ruc)
with open(f'{OUT}vars.json', encoding='utf-8') as f: VARS = json.load(f)

def muestra_comparacion(y, anios, semilla=SEM_MUESTRA, n_por_anio=N_MUESTRA):
    r = np.random.default_rng(semilla); partes = []
    for a in np.unique(anios):
        m = np.where((y == 0) & (anios == a))[0]
        if len(m): partes.append(r.choice(m, size=min(n_por_anio, len(m)), replace=False))
    return np.concatenate(partes)

def estadistico(X, y, gi, anios, n_folds, k, semilla, Xu=None):
    """Percentil mediano dentro de cada anio de los positivos. `gi` son codigos enteros de conglomerado.
    Cada positivo lo puntua un modelo que NO vio su conglomerado; la poblacion de comparacion la
    puntua el PROMEDIO de todos los pliegues. En el Paso 7 la puntuaba uno solo: eso era una
    asimetria entre positivos y comparacion, y aqui queda corregida."""
    r  = np.random.default_rng(semilla)
    gp = np.unique(gi[y == 1])
    if len(gp) < 2: return np.nan
    idx_s = np.concatenate([np.where(y == 1)[0], muestra_comparacion(y, anios)])
    Xs = X[idx_s]
    if Xu is None: Xu = X[y == 0]        # fuera del bucle: copiar la matriz en cada pliegue era el cuello
    score = np.full(len(idx_s), np.nan); acum = np.zeros(len(idx_s)); nf = 0
    folds = ([[c] for c in gp] if n_folds is None
             else [f for f in np.array_split(r.permutation(gp), min(n_folds, len(gp))) if len(f)])
    for j, fuera in enumerate(folds):
        dentro_fold = np.isin(gi, fuera)
        s = pu_bagging(X[(y == 1) & ~dentro_fold], Xu, Xs, k=k, seed=semilla + j)
        donde = dentro_fold[idx_s] & (y[idx_s] == 1)
        score[donde] = s[donde]
        acum += s; nf += 1
    falta = np.isnan(score); score[falta] = (acum / nf)[falta]
    ev  = pd.DataFrame({'y': y[idx_s], 'anio': anios[idx_s], 'm': score})
    pct = ev.groupby('anio')['m'].rank(pct=True).values
    return float(np.nanmedian(pct[ev.y.values == 1]))

def conglomerados_placebo(tam, pos, r):
    """Elige tantos conglomerados no positivos como positivos haya, con el MISMO numero de filas.
    Sin emparejar, los placebos serian proveedores de un solo contrato y el nulo saldria sesgado."""
    base = tam.drop(index=[c for c in pos if c in tam.index])
    if len(base) == 0: return np.array([], dtype=int)
    libres = base.copy(); sel = []
    for c in pos:
        if len(libres) == 0: libres = base.copy()   # universo chico: se permite repetir conglomerado
        n = int(tam[c]); v = libres.values
        cand = libres.index[v == n]
        if len(cand) == 0:
            dd = np.abs(v - n); cand = libres.index[dd == dd.min()]
        e = r.choice(np.asarray(cand)); sel.append(e)
        if e in libres.index: libres = libres.drop(index=e)
    return np.asarray(sel)

def permutacion(nombre, d, col_y, col_g, cols, n_folds=None, n_perm=N_PERM, k=K_P):
    y = d[col_y].values.astype(int)
    gi = pd.factorize(d[col_g].values)[0]          # codigos enteros: isin sobre texto era caro
    anios = d.anio_proceso.values
    X = d[cols].astype(float).values
    gp = np.unique(gi[y == 1]); tam = pd.Series(gi).value_counts()
    print(f'\n{"="*68}\n{nombre}\n  filas {len(d):,} | positivos {y.sum()} en {len(gp)} conglomerados | {len(cols)} variables')
    pctix = {nom: d.groupby('anio_proceso')[c].rank(pct=True).values
             for nom, c in [('cri_v2','cri2_medio'), ('idx_intermediacion','idx_intermediacion')]
             if c in d.columns}
    Xu  = X[y == 0]
    t0  = time.time()
    obs = {'modelo': estadistico(X, y, gi, anios, n_folds, k, SEMILLA, Xu=Xu)}
    print(f'  estadistico observado: {obs["modelo"]:.3f} ({time.time()-t0:.0f} s)')
    for nom, pz in pctix.items(): obs[nom] = float(np.nanmedian(pz[y == 1]))
    t0 = time.time(); nul = {m: [] for m in obs}
    for b in range(n_perm):
        r   = np.random.default_rng(SEMILLA + 100000 + b)
        sel = conglomerados_placebo(tam, gp, r)
        yp  = np.isin(gi, sel).astype(int)
        nul['modelo'].append(estadistico(X, yp, gi, anios, n_folds, k, SEMILLA + 100000 + b,
                                         Xu=X[(y == 0) & (yp == 0)]))
        for nom, pz in pctix.items(): nul[nom].append(float(np.nanmedian(pz[yp == 1])))
        print(f'    {b+1}/{n_perm} permutaciones | {time.time()-t0:.0f} s', end='\r')
    print(f'    {n_perm} permutaciones en {time.time()-t0:.0f} s' + ' ' * 25)
    filas = []
    for m, o in obs.items():
        v = np.array([x for x in nul[m] if np.isfinite(x)])
        if len(v) < 10:
            print(f'  AVISO {m}: solo {len(v)} permutaciones validas (casi todos los conglomerados')
            print('  son positivos, no hay de donde sacar placebos). No se reporta valor p.')
            filas.append((nombre, m, round(o, 3), np.nan, np.nan, np.nan, np.nan, np.nan, len(v)))
            continue
        med = float(np.median(v))
        p_der = (1 + int((v >= o).sum())) / (len(v) + 1)
        p_dos = (1 + int((np.abs(v - med) >= abs(o - med)).sum())) / (len(v) + 1)
        filas.append((nombre, m, round(o, 3), round(med, 3),
                      round(float(np.percentile(v, 2.5)), 3), round(float(np.percentile(v, 97.5)), 3),
                      round(p_der, 4), round(p_dos, 4), len(v)))
    t = pd.DataFrame(filas, columns=['caso','metodo','observado','nulo_mediana','nulo_p2.5',
                                     'nulo_p97.5','p_una_cola','p_dos_colas','n_perm'])
    print(t.drop(columns='caso').to_string(index=False))
    return t

uA = P.aparece_oferentes.values.astype(bool)
uB = (P.tiene_grupo & P.aparece_oferentes).values.astype(bool)
casos = [
    ('A fantasma · completo',           P,      'y_fantasma',   'proveedor_ruc', VARS['A'],            None),
    ('A fantasma · universo oferentes', P[uA],  'y_fantasma',   'proveedor_ruc', VARS['A'],            None),
    ('A fantasma · anticipada',         P,      'y_fantasma',   'proveedor_ruc', VARS['A_anticipada'], None),
    ('B intragrupo · completo',         P[uB],  'y_intragrupo', 'grupo_B',       VARS['B'],            5),
    ('B intragrupo · anticipada',       P[uB],  'y_intragrupo', 'grupo_B',       VARS['B_anticipada'], 5),
]
perm = pd.concat([permutacion(n, d, cy, cg, cs, nf) for n, d, cy, cg, cs, nf in casos], ignore_index=True)
perm.to_csv(f'{OUT}permutacion.csv', index=False)
print(f'\n  -> {OUT}permutacion.csv')
print('\nComo se lee: el nulo ya NO esta centrado en 0,50, porque validar dejando un conglomerado')
print('fuera desplaza el estadistico por si solo. Eso explica el control negativo del 11.2 y es')
print('justamente lo que esta prueba corrige. El valor p que se reporta es p_una_cola.')
anotar('12_permutacion', n_perm=N_PERM, k_bolsas=K_P, n_muestra=N_MUESTRA, tabla=perm.to_dict('records'))

In [ ]:
# ============ PASO 13 · Cobertura, universo restringido y varianza de los pliegues ============
# Usa `estadistico` del Paso 12, asi que el Paso 12 tiene que haberse corrido antes.
# Responde tres cosas que el Paso 11 dejo abiertas:
#   13.1 el 0,757 de "solo ausencia" del objetivo A: ¿se cae si el universo se iguala en cobertura?
#   13.2 el objetivo B dio 0,806 / 0,859 / 0,878 en tres pasos distintos: ¿cuanto de eso es la
#        particion de pliegues y cuanto es el modelo?
#   13.3 ¿cuanto de la senal del objetivo A es la provincia (Guayas) y nada mas?

K_13, R_PART = 25, 10

def partes(d, col_y, col_g, cols):
    return (d[cols].astype(float).values, d[col_y].values.astype(int),
            pd.factorize(d[col_g].values)[0], d.anio_proceso.values)

uA = P.aparece_oferentes.values.astype(bool)
uB = (P.tiene_grupo & P.aparece_oferentes).values.astype(bool)

# ---------- 13.1 Valores reales contra solo ausencia, en los dos universos ----------
print('='*68 + '\n13.1  Comportamiento o cobertura, dentro y fuera del universo de oferentes\n' + '='*68)
f13 = []
casos13 = [('A · panel completo',      P,     'y_fantasma',   'proveedor_ruc', VARS['A'], None),
           ('A · universo oferentes',  P[uA], 'y_fantasma',   'proveedor_ruc', VARS['A'], None),
           ('B · universo oferentes',  P[uB], 'y_intragrupo', 'grupo_B',       VARS['B'], 5)]
for nom, d, cy, cg, cols, nf in casos13:
    X, y, gi, an = partes(d, cy, cg, cols)
    Xm = np.isnan(X).astype(float); Xm = Xm[:, Xm.std(axis=0) > 0]
    o_v = estadistico(X,  y, gi, an, nf, K_13, SEMILLA)
    o_m = estadistico(Xm, y, gi, an, nf, K_13, SEMILLA)
    f13.append((nom, len(d), int(y.sum()), round(o_v,3), round(o_m,3), round(o_v-o_m,3)))
    print(f'  {nom:24s} filas {len(d):>7,} | valores {o_v:.3f} | solo ausencia {o_m:.3f} | distancia {o_v-o_m:+.3f}')
t13a = pd.DataFrame(f13, columns=['caso','filas','positivos','valores','solo_ausencia','distancia'])
print('\nSi dentro del universo de oferentes la distancia se abre, la parte de "cobertura" del')
print('resultado del panel completo era un artefacto de quien aparece o no en la tabla de oferentes.')

# ---------- 13.2 Cuanto mueve la particion de pliegues ----------
print('\n' + '='*68 + f'\n13.2  Varianza por particion de pliegues (B, {R_PART} particiones de 5 pliegues)\n' + '='*68)
X, y, gi, an = partes(P[uB], 'y_intragrupo', 'grupo_B', VARS['B'])
vals = [estadistico(X, y, gi, an, 5, K_13, SEMILLA + 500 + i) for i in range(R_PART)]
vals = np.array([v for v in vals if np.isfinite(v)])
n_gp = len(np.unique(gi[y == 1]))
print(f'  5 pliegues, {len(vals)} particiones: media {vals.mean():.3f} | min {vals.min():.3f} | '
      f'max {vals.max():.3f} | rango {vals.max()-vals.min():.3f}')
# dejar uno fuera cuesta un pliegue por conglomerado: solo tiene sentido si no son demasiados
uno = estadistico(X, y, gi, an, None, K_13, SEMILLA) if n_gp <= 80 else np.nan
print(f'  dejando un conglomerado fuera ({n_gp} pliegues, sin particion arbitraria): ' +
      (f'{uno:.3f}' if np.isfinite(uno) else 'no se corre, demasiados conglomerados'))
t13b = pd.DataFrame({'particion': range(len(vals)), 'mediana_percentil': np.round(vals,3)})
print('\nLa cifra que se reporta de B es la de "un conglomerado fuera": no depende de como se parta.')
print('El rango de arriba es la incertidumbre que hay que declarar si se reporta la version de 5 pliegues.')

# ---------- 13.3 ¿Es solo la provincia? ----------
print('\n' + '='*68 + '\n13.3  Aporte de la geografia en el objetivo A\n' + '='*68)
# solo los indicadores de ubicacion. No se mete sup_frac_fuera_provincia ni nada parecido:
# eso es conducta (contratar lejos), no geografia.
geo = [c for c in VARS['A'] if c.startswith('sup_') and c.split('sup_')[-1] in
       ('guayas','pichincha','azuay','manabi','el_oro','tungurahua')]
X, y, gi, an = partes(P, 'y_fantasma', 'proveedor_ruc', VARS['A'])
sin_geo = [c for c in VARS['A'] if c not in geo]
o_todo = estadistico(X, y, gi, an, None, K_13, SEMILLA)
Xg, _, _, _ = partes(P, 'y_fantasma', 'proveedor_ruc', sin_geo)
o_sin  = estadistico(Xg, y, gi, an, None, K_13, SEMILLA)
Xs_, _, _, _ = partes(P, 'y_fantasma', 'proveedor_ruc', geo)
o_solo = estadistico(Xs_, y, gi, an, None, K_13, SEMILLA)
print(f'  variables geograficas detectadas ({len(geo)}): {geo}')
print(f'  con todo           : {o_todo:.3f}')
print(f'  sin geografia      : {o_sin:.3f}   (caida {o_todo-o_sin:+.3f})')
print(f'  solo geografia     : {o_solo:.3f}')
t13c = pd.DataFrame([('con todo',o_todo),('sin geografia',o_sin),('solo geografia',o_solo)],
                    columns=['variables','mediana_percentil']).round(3)

t13a.to_csv(f'{OUT}cobertura_universo.csv', index=False)
t13b.to_csv(f'{OUT}varianza_pliegues.csv', index=False)
t13c.to_csv(f'{OUT}aporte_geografia.csv', index=False)
print(f'\n  -> {OUT}cobertura_universo.csv, {OUT}varianza_pliegues.csv, {OUT}aporte_geografia.csv')
anotar('13_cobertura', K_13=K_13, R_PART=R_PART, cobertura=t13a.to_dict('records'),
       pliegues=dict(media=float(vals.mean()), min=float(vals.min()), max=float(vals.max()), uno_fuera=float(uno)),
       geografia=t13c.to_dict('records'))

In [ ]:
# ============ CHEQUEO · el catastro del RUC que ya quedo consolidado ============
# El archivo c8_catastro_ruc.parquet ya existe (107 MB), asi que la celda RUC-B corrio.
# Pero corrio con ENC = 'latin-1', y RUC-A demostro que los CSV estan en utf-8: si fuera asi,
# los nombres con tilde o con ñ quedaron destrozados. Esto lo confirma o lo descarta en 10 s.

import unicodedata
CR = f'{OUT}catastro_ruc.parquet'
cat = pd.read_parquet(CR)
print(f'{CR}: {len(cat):,} filas x {cat.shape[1]} columnas')
print('Columnas:', list(cat.columns))

txt = [c for c in cat.columns if cat[c].dtype == object or pd.api.types.is_string_dtype(cat[c])]
MOJI = ('Ã', 'Â', 'ï¿½', '�')
print('\n--- Senales de codificacion mal leida ---')
malo = 0
for c in txt[:20]:
    s = cat[c].astype('string').dropna()
    if not len(s): continue
    n = int(s.str.contains('|'.join(MOJI), regex=True, na=False).sum())
    malo += n
    if n: print(f'  {c:32s} {n:>8,} valores con basura de codificacion')
print('  ninguna' if malo == 0 else f'  TOTAL: {malo:,} valores afectados -> hay que volver a correr RUC-B con utf-8')

for c in [x for x in ('RAZON_SOCIAL','NOMBRE_FANTASIA_COMERCIAL','ACTIVIDAD_ECONOMICA',
                      'DESCRIPCION_PROVINCIA_EST','DESCRIPCION_CANTON_EST') if x in cat.columns][:3]:
    ej = cat[c].astype('string').dropna()
    ej = ej[ej.str.contains('[ÑÁÉÍÓÚáéíóúñ]', regex=True, na=False)]
    print(f'\n  {c} · ejemplos con tilde o ñ ({len(ej):,}):')
    for v in ej.head(4).tolist(): print('   ', v)

print('\n--- Cobertura sobre los proveedores de la SERCOP ---')
rucs = con.sql('SELECT DISTINCT proveedor_ruc FROM bmv').df().proveedor_ruc
k = [c for c in cat.columns if c.upper() in ('NUMERO_RUC','RUC')][0]
en = set(cat[k].astype(str))
print(f'  proveedores distintos en la SERCOP: {len(rucs):,}')
print(f'  de esos, presentes en el catastro:  {sum(r in en for r in rucs):,} '
      f'({sum(r in en for r in rucs)/len(rucs)*100:.1f}%)')

print('\n--- Fechas que nos interesan ---')
for c in [x for x in ('FECHA_INICIO_ACTIVIDADES','FECHA_SUSPENSION_DEFINITIVA',
                      'FECHA_REINICIO_ACTIVIDADES','FECHA_CESE') if x in cat.columns]:
    f = pd.to_datetime(cat[c], errors='coerce')
    print(f'  {c:30s} validas {f.notna().mean()*100:5.1f}% | '
          f'{str(f.min())[:10]} .. {str(f.max())[:10]}')

In [ ]:
# ============ RUC-B v2 · reconsolidar el catastro con la codificacion correcta ============
# El chequeo encontro 2.127.608 valores con basura de codificacion y CERO nombres con tilde o
# con ñ: el archivo anterior se leyo como latin-1 cuando los CSV son utf-8. Hay que rehacerlo.
# De paso resuelve el otro problema: solo el 22,0% de los proveedores de la SERCOP aparecian en
# el catastro, lo que casi siempre es la llave mal armada y no una falta real de datos.

import glob, os
RUTA_SRI = r'C:\SERCOP_FINAL\05_documentos\SRI'
SEP, ENC = '|', 'utf-8'                     # utf-8: lo demostro RUC-A
GLOB = (RUTA_SRI + r'\*.csv').replace('\\', '/')
n_arch = len(glob.glob(GLOB))
assert n_arch > 0, f'No hay CSV en {RUTA_SRI}'
print(f'{n_arch} archivos en {RUTA_SRI}')

# ---------- 1. Leer en crudo, todo como texto, sin filtrar nada ----------
t0 = time.time()
con.sql(f"""
CREATE OR REPLACE VIEW cat_crudo AS
SELECT * FROM read_csv('{GLOB}', delim='{SEP}', header=true, encoding='{ENC}',
                       all_varchar=true, union_by_name=true, ignore_errors=true)
""")
cols = [r[0] for r in con.sql('DESCRIBE cat_crudo').fetchall()]
n_crudo = con.sql('SELECT count(*) FROM cat_crudo').fetchone()[0]
print(f'Catastro crudo: {n_crudo:,} filas x {len(cols)} columnas en {time.time()-t0:.0f} s')

COL_RUC = next(c for c in cols if c.upper() in ('NUMERO_RUC','RUC'))
print(f'Columna de RUC: {COL_RUC}')
print('\n--- Como viene el RUC ---')
print(con.sql(f"""
SELECT length(trim({COL_RUC})) AS largo, count(*) AS filas,
       min(trim({COL_RUC})) AS ejemplo_min, max(trim({COL_RUC})) AS ejemplo_max
FROM cat_crudo GROUP BY 1 ORDER BY filas DESC
""").df().to_string(index=False))

# ---------- 2. Armar la llave de 13 y comparar las dos opciones ----------
con.sql(f"""
CREATE OR REPLACE VIEW cat13 AS
SELECT *, trim({COL_RUC}) AS ruc_tal_cual, lpad(trim({COL_RUC}), 13, '0') AS ruc13
FROM cat_crudo
""")
d = con.sql("""
SELECT (SELECT count(DISTINCT proveedor_ruc) FROM bmv)                                   AS sercop,
       (SELECT count(DISTINCT p.proveedor_ruc) FROM (SELECT DISTINCT proveedor_ruc FROM bmv) p
          WHERE p.proveedor_ruc IN (SELECT ruc_tal_cual FROM cat13))                     AS con_tal_cual,
       (SELECT count(DISTINCT p.proveedor_ruc) FROM (SELECT DISTINCT proveedor_ruc FROM bmv) p
          WHERE p.proveedor_ruc IN (SELECT ruc13 FROM cat13))                            AS con_lpad13
""").df().iloc[0]
print('\n--- Cobertura segun la llave que se use ---')
print(f'  proveedores distintos en la SERCOP : {int(d.sercop):,}')
print(f'  encontrados con el RUC tal cual    : {int(d.con_tal_cual):,} ({d.con_tal_cual/d.sercop*100:5.1f}%)')
print(f'  encontrados rellenando a 13 con 0  : {int(d.con_lpad13):,} ({d.con_lpad13/d.sercop*100:5.1f}%)')
LLAVE = 'ruc13' if d.con_lpad13 >= d.con_tal_cual else 'ruc_tal_cual'
print(f'  -> se usa {LLAVE}')
if max(d.con_lpad13, d.con_tal_cual) / d.sercop < 0.70:
    print('  AVISO: aun asi falta mas del 30%. Puede que falten provincias en la descarga o que')
    print('  el catastro solo traiga contribuyentes activos. Hay que declararlo como limite del dato.')

# ---------- 3. Guardar solo los RUC que nos sirven, con fechas saneadas ----------
FECHAS = [c for c in cols if c.upper().startswith('FECHA')]
sel_f = ', '.join([f"""CASE WHEN try_cast({c} AS DATE) BETWEEN DATE '1900-01-01' AND DATE '2026-12-31'
                       THEN try_cast({c} AS DATE) END AS {c}""" for c in FECHAS])
otras = ', '.join([c for c in cols if c not in FECHAS])
t0 = time.time()
con.sql(f"""
COPY (
  SELECT {LLAVE} AS ruc, {otras}, {sel_f}
  FROM cat13
  WHERE {LLAVE} IN (SELECT DISTINCT proveedor_ruc FROM bmv)
) TO '{R}/c8_catastro_ruc_tmp.parquet' (FORMAT parquet, COMPRESSION zstd)
""")
os.replace(f'{R}/c8_catastro_ruc_tmp.parquet', f'{OUT}catastro_ruc.parquet')
cat = pd.read_parquet(f'{OUT}catastro_ruc.parquet')
print(f'\n  -> {OUT}catastro_ruc.parquet ({len(cat):,} filas x {cat.shape[1]} cols) en {time.time()-t0:.0f} s')
print(f'  RUC distintos: {cat.ruc.nunique():,} | establecimientos por RUC: '
      f'{len(cat)/max(cat.ruc.nunique(),1):.1f} en promedio')

# ---------- 4. Verificar que la codificacion quedo bien ----------
MOJI = ('Ã', 'Â', 'ï¿½', '\ufffd')
print('\n--- Verificacion de codificacion ---')
malo = 0
for c in [x for x in cat.columns if pd.api.types.is_string_dtype(cat[x]) or cat[x].dtype == object]:
    s = cat[c].astype('string').dropna()
    if not len(s): continue
    n = int(s.str.contains('|'.join(MOJI), regex=True, na=False).sum())
    if n: print(f'  {c:32s} {n:>8,} valores con basura'); malo += n
print('  ninguna: la codificacion quedo bien' if malo == 0 else f'  TODAVIA hay {malo:,} valores malos')
for c in [x for x in ('RAZON_SOCIAL','ACTIVIDAD_ECONOMICA','DESCRIPCION_CANTON_EST') if x in cat.columns]:
    ej = cat[c].astype('string').dropna()
    ej = ej[ej.str.contains('[ÑÁÉÍÓÚáéíóúñ]', regex=True, na=False)]
    print(f'  {c} con tilde o ñ: {len(ej):,}' + (f' | ej: {ej.iloc[0]}' if len(ej) else ''))

print('\n--- Fechas despues de sanear ---')
for c in FECHAS:
    if c not in cat.columns: continue
    f = pd.to_datetime(cat[c], errors='coerce')
    print(f'  {c:30s} validas {f.notna().mean()*100:5.1f}% | {str(f.min())[:10]} .. {str(f.max())[:10]}')
anotar('ruc_b2', archivos=n_arch, encoding=ENC, llave=LLAVE, filas_crudo=int(n_crudo),
       filas_guardadas=len(cat), rucs=int(cat.ruc.nunique()),
       cobertura=float(max(d.con_lpad13, d.con_tal_cual) / d.sercop))

In [ ]:
# ============ PASO 14 · De donde viene la senal de la ausencia ============
# El 13.1 dejo claro que el modelo construido SOLO con indicadores de ausencia llega a 0.737 (A)
# y 0.714 (B), y que restringir al universo de oferentes NO cierra esa brecha. Antes de defender
# el resultado hay que saber QUE ausencias concretas estan hablando, y si cada una es un artefacto
# administrativo (quien figura en tal registro) o es conducta (no tuvo con quien competir).
# Requiere el Paso 12 corrido: usa `estadistico` y `permutacion`.

UMBRAL_AUS = 0.15      # |AUC-0.5| de la ausencia por encima de esto = ausencia informativa

with open(f'{OUT}familias.json', encoding='utf-8') as f: fam = json.load(f)
de_familia = {c: f for f, cs in fam.items() for c in cs}
uA = P.aparece_oferentes.values.astype(bool)
PA = P[uA]
yA = PA.y_fantasma.values.astype(int)

print('='*68 + '\n14.1  Que ausencias discriminan (objetivo A, universo de oferentes)\n' + '='*68)
filas = []
for c in VARS['A']:
    na = pd.isna(PA[c]).values.astype(float)
    if na.std() == 0: continue
    filas.append((c, de_familia.get(c, '?'), round(auc(na, yA), 3), round(float(na.mean()), 3)))
aus = pd.DataFrame(filas, columns=['variable','familia','auc_ausencia','pct_nulos'])
aus['fuerza'] = (aus.auc_ausencia - 0.5).abs().round(3)
aus = aus.sort_values('fuerza', ascending=False)
print(aus.head(20).to_string(index=False))
print(f'\nVariables con ausencia informativa (|AUC-0.5| > {UMBRAL_AUS}): {int((aus.fuerza > UMBRAL_AUS).sum())} de {len(aus)}')
print('\n--- Resumen por familia ---')
res = aus.groupby('familia').agg(n_vars=('variable','size'), fuerza_media=('fuerza','mean'),
                                 fuerza_max=('fuerza','max'), informativas=('fuerza', lambda s: int((s > UMBRAL_AUS).sum())))
print(res.sort_values('fuerza_media', ascending=False).round(3).to_string())
print('\nauc_ausencia > 0.5 = los positivos tienen MAS nulos en esa variable.')
print('auc_ausencia < 0.5 = los positivos tienen MENOS nulos, o sea mas historial que el resto.')

# ---------- 14.2 Conjunto limpio: solo variables cuya ausencia no dice nada ----------
sucias = set(aus.loc[aus.fuerza > UMBRAL_AUS, 'variable'])
if len(sucias) == 0:
    sucias = set(aus.head(15).variable)
    print('\nNinguna variable pasa el umbral por si sola: la senal de la ausencia esta repartida')
    print('entre muchas ausencias debiles, no concentrada en una. Eso ya es una respuesta, pero para')
    print('la prueba del 14.3 se quitan de todas formas las 15 mas fuertes.')
VARS['A_sin_ausencia'] = [c for c in VARS['A'] if c not in sucias]
VARS['B_sin_ausencia'] = [c for c in VARS['B'] if c not in sucias]
with open(f'{OUT}vars.json','w',encoding='utf-8') as f: json.dump(VARS, f, ensure_ascii=False, indent=1)
print(f"\nConjunto A_sin_ausencia: {len(VARS['A_sin_ausencia'])} variables "
      f"(se quitaron {len(VARS['A'])-len(VARS['A_sin_ausencia'])})")
print(f"Conjunto B_sin_ausencia: {len(VARS['B_sin_ausencia'])} variables "
      f"(se quitaron {len(VARS['B'])-len(VARS['B_sin_ausencia'])})")
aus.to_csv(f'{OUT}ausencias.csv', index=False)

# ---------- 14.3 ¿Aguanta el modelo sin esas variables? ----------
print('\n' + '='*68 + '\n14.3  El modelo con las variables cuya ausencia no informa\n' + '='*68)
print('Si aqui sigue superando a su nulo, la objecion de "esta detectando cobertura" queda cerrada.')
uB = (P.tiene_grupo & P.aparece_oferentes).values.astype(bool)
casos14 = [
    ('A fantasma · sin ausencia informativa', P[uA], 'y_fantasma',   'proveedor_ruc', VARS['A_sin_ausencia'], None),
    ('B intragrupo · sin ausencia informativa', P[uB], 'y_intragrupo', 'grupo_B',     VARS['B_sin_ausencia'], 5),
]
p14 = pd.concat([permutacion(n, d, cy, cg, cs, nf) for n, d, cy, cg, cs, nf in casos14], ignore_index=True)
p14.to_csv(f'{OUT}permutacion_sin_ausencia.csv', index=False)
print(f'\n  -> {OUT}ausencias.csv, {OUT}permutacion_sin_ausencia.csv, {OUT}vars.json (actualizado)')
anotar('14_ausencias', umbral=UMBRAL_AUS, quitadas=sorted(sucias),
       n_A=len(VARS['A_sin_ausencia']), n_B=len(VARS['B_sin_ausencia']), tabla=p14.to_dict('records'))

In [ ]:
# ============ PASO 15 · Figuras para la tesis ============
# Cinco figuras, cada una con el CSV que la genera al lado, en 03_resultados/figuras/.
# Requiere el Paso 12 corrido: usa `estadistico`, `P` y `VARS`.
try:
    import matplotlib
except ImportError:
    import subprocess; subprocess.run([sys.executable,'-m','pip','install','matplotlib'], check=True)
    import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

FIG = f'{R}/figuras'; os.makedirs(FIG, exist_ok=True)
AZUL, NARANJA, AGUA = '#2a78d6', '#eb6834', '#1baf7a'
TINTA, TINTA2, GRIS, BANDA = '#0b0b0b', '#52514e', '#b9b8b4', '#e4e3df'
plt.rcParams.update({'font.size':9, 'axes.edgecolor':GRIS, 'axes.labelcolor':TINTA2,
                     'xtick.color':TINTA2, 'ytick.color':TINTA2, 'axes.titlesize':10,
                     'figure.dpi':300, 'savefig.dpi':300, 'savefig.bbox':'tight',
                     'axes.spines.top':False, 'axes.spines.right':False, 'axes.grid':False})
COMA = FuncFormatter(lambda v, _: f'{v:.2f}'.replace('.', ','))
def c(x, n=2): return f'{x:.{n}f}'.replace('.', ',')
def guarda(fig, nombre, datos=None):
    fig.savefig(f'{FIG}/{nombre}.png'); plt.close(fig)
    if datos is not None: datos.to_csv(f'{FIG}/{nombre}.csv', index=False)
    print(f'  -> {FIG}/{nombre}.png' + (' + .csv' if datos is not None else ''))

ETIQ  = {'modelo':'Modelo aprendido', 'idx_intermediacion':'Índice de intermediación', 'cri_v2':'Índice tipo CRI'}
COLOR = {'modelo':AZUL, 'idx_intermediacion':AGUA, 'cri_v2':NARANJA}
ORDEN = ['modelo','idx_intermediacion','cri_v2']

# ===================================================================
# FIGURA 1 · Observado contra el nulo de permutacion, por especificacion
# ===================================================================
pm = pd.read_csv(f'{OUT}permutacion.csv')
try:    pm = pd.concat([pm, pd.read_csv(f'{OUT}permutacion_sin_ausencia.csv')], ignore_index=True)
except FileNotFoundError: pass
pm = pm.dropna(subset=['nulo_mediana'])
assert len(pm), 'c8_permutacion.csv no tiene ninguna fila con nulo valido: corre el Paso 12 primero'

ORDEN_CASO = ['A fantasma · completo', 'A fantasma · universo oferentes',
              'A fantasma · sin ausencia informativa', 'A fantasma · anticipada',
              'B intragrupo · completo', 'B intragrupo · sin ausencia informativa']
casos = [c for c in ORDEN_CASO if c in set(pm.caso)] + [c for c in pm.caso.unique() if c not in ORDEN_CASO]
fs, cab = [], []                       # filas del grafico y cuales son cabecera de grupo
for caso in casos:
    fs.append(None); cab.append(caso)
    for m in ORDEN:
        r = pm[(pm.caso == caso) & (pm.metodo == m)]
        if len(r): fs.append(r.iloc[0]); cab.append('   ' + ETIQ[m])
y = np.arange(len(fs))[::-1]

fig, ax = plt.subplots(figsize=(6.3, 0.265*len(fs) + 1.25))
ax.axvline(0.5, color=GRIS, lw=0.8, ls=':', zorder=0)
for i, r in enumerate(fs):
    if r is None: continue
    ax.plot([r['nulo_p2.5'], r['nulo_p97.5']], [y[i], y[i]], lw=5.5, color=BANDA,
            solid_capstyle='round', zorder=1)
    ax.plot([r['nulo_mediana']], [y[i]], marker='|', ms=7, color=TINTA2, mew=1.2, zorder=2)
    ax.plot([r['observado']], [y[i]], marker='o', ms=6, color=COLOR[r['metodo']],
            mec='white', mew=1.2, zorder=3)
    sig = r['p_una_cola'] <= 0.05
    ax.text(1.015, y[i], 'p = ' + c(r['p_una_cola']), va='center', ha='left', fontsize=7.5,
            color=TINTA if sig else TINTA2, fontweight='bold' if sig else 'normal')
ax.set_yticks(y); ax.set_yticklabels(cab, fontsize=8)
for t, lab in zip(ax.get_yticklabels(), cab):
    if not lab.startswith('   '): t.set_fontweight('bold'); t.set_color(TINTA)
ax.tick_params(axis='y', length=0)
izq = min(min(r['nulo_p2.5'] for r in fs if r is not None),
          min(r['observado'] for r in fs if r is not None)) - 0.05
ax.set_xlim(izq, 1.0); ax.set_ylim(y.min()-0.7, y.max()+0.7)
ax.set_xlabel('Percentil mediano de los positivos dentro del año')
ax.xaxis.set_major_formatter(COMA)
h = [plt.Line2D([],[],marker='o',ls='',ms=6,mec='white',mew=1.2,color=COLOR[m],label=ETIQ[m]) for m in ORDEN]
h += [plt.Line2D([],[],lw=5.5,color=BANDA,label='Intervalo 95 % del nulo'),
      plt.Line2D([],[],marker='|',ls='',ms=7,mew=1.2,color=TINTA2,label='Centro del nulo'),
      plt.Line2D([],[],ls=':',lw=0.8,color=GRIS,label='0,50 (azar sin validación)')]
ax.legend(handles=h, loc='upper center', bbox_to_anchor=(0.5, -0.09/(0.265*len(fs)+1.25)*8),
          ncol=2, frameon=False, fontsize=7.5)
guarda(fig, 'fig1_observado_vs_nulo',
       pm[['caso','metodo','observado','nulo_mediana','nulo_p2.5','nulo_p97.5','p_una_cola']])

# ===================================================================
# FIGURA 2 · Perfil de posicion en la red de las empresas del catastro del SRI
# ===================================================================
# Percentiles dentro de estratos ano x decil de actividad: 0,50 = indistinguible del resto.
PERFIL = [('red_autovector','Centralidad de autovector'), ('monto_anio','Monto contratado en el año'),
          ('cob_n_rivales','Rivales distintos enfrentados'), ('red_coreness','Pertenencia al núcleo (k-core)'),
          ('com_grado_total','Conexiones entre comunidades'), ('n_contracts','Número de contratos'),
          ('cob_hhi_rivales','Concentración de rivales (HHI)'), ('red_hhi_entidades','Concentración de compradores (HHI)'),
          ('cob_agrupamiento','Coeficiente de agrupamiento')]
faltan = [v for v, _ in PERFIL if v not in P.columns]
PERFIL = [(v, e) for v, e in PERFIL if v in P.columns]
if faltan: print(f'  Figura 2: no estan en el panel y se omiten: {faltan}')
act = 'n_contracts' if 'n_contracts' in P.columns else PERFIL[0][0]
print(f'  Figura 2: estrato de actividad = {act} | {len(PERFIL)} indicadores')
cols_q = list(dict.fromkeys(['anio_proceso','y_fantasma', act] + [v for v, _ in PERFIL]))
Q = P[cols_q].copy()
Q['decil'] = Q.groupby('anio_proceso')[act].transform(
    lambda s: pd.qcut(s.rank(method='first'), 10, labels=False, duplicates='drop'))
f2 = []
for v, e in PERFIL:
    pct = Q.groupby(['anio_proceso','decil'])[v].rank(pct=True)
    f2.append((e, v, float(np.nanmedian(pct[Q.y_fantasma == 1]))))
d2 = pd.DataFrame(f2, columns=['indicador','variable','percentil']).sort_values('percentil')

fig, ax = plt.subplots(figsize=(6.3, 0.33*len(d2) + 0.9))
yy = np.arange(len(d2))
ax.barh(yy, d2.percentil - 0.5, left=0.5, height=0.6,
        color=[AZUL if v > 0.5 else NARANJA for v in d2.percentil], zorder=2)
ax.axvline(0.5, color=TINTA2, lw=1.0, zorder=3)
for i, v in enumerate(d2.percentil):
    ax.text(v + (0.012 if v > 0.5 else -0.012), yy[i], c(v, 3), va='center',
            ha='left' if v > 0.5 else 'right', fontsize=7.5, color=TINTA2)
ax.set_yticks(yy); ax.set_yticklabels(d2.indicador, fontsize=8)
ax.tick_params(axis='y', length=0)
ax.set_xlim(0, 1); ax.set_xlabel('Percentil dentro del estrato año × decil de actividad')
ax.xaxis.set_major_formatter(COMA); ax.set_ylim(-0.7, len(d2)-0.3)
ax.text(0.5, len(d2)-0.45, '0,50 = indistinguible de un proveedor comparable',
        ha='center', va='bottom', fontsize=7.5, color=TINTA2)
guarda(fig, 'fig2_perfil_red', d2)

# ===================================================================
# FIGURA 3 · Distribucion acumulada del percentil: positivos frente a la poblacion
# ===================================================================
sc = pd.read_parquet(f'{OUT}scores.parquet')
sc['pct'] = sc.groupby(['objetivo','anio'])['modelo'].rank(pct=True)
fig, axs = plt.subplots(1, 2, figsize=(6.3, 2.7), sharey=True)
f3 = []
for ax, (obj, tit) in zip(axs, [('A','Objetivo A · empresas del catastro del SRI'),
                                ('B','Objetivo B · grupos del mismo representante')]):
    s = sc[sc.objetivo == obj]
    for yv, col, lab in [(0, GRIS, 'Población de comparación'), (1, AZUL, 'Positivos')]:
        v = np.sort(s.loc[s.y == yv, 'pct'].dropna().values)
        if not len(v): continue
        ax.step(v, np.arange(1, len(v)+1)/len(v), where='post', color=col,
                lw=2.0 if yv else 1.4, label=lab, zorder=3 if yv else 2)
        rej = np.linspace(0, 1, 101)       # el csv va resumido en una rejilla, no punto por punto
        f3 += [(obj, lab, round(float(x), 2), round(float((v <= x).mean()), 4)) for x in rej]
    ax.plot([0,1],[0,1], ls=':', lw=0.8, color=GRIS, zorder=1)
    ax.set_title(tit, fontsize=8.5, color=TINTA)
    ax.set_xlabel('Percentil del puntaje dentro del año')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)
axs[0].set_ylabel('Proporción acumulada')
axs[0].legend(loc='upper left', frameon=False, fontsize=7.5)
guarda(fig, 'fig3_distribucion_percentiles',
       pd.DataFrame(f3, columns=['objetivo','grupo','percentil','proporcion_acumulada']))

# ===================================================================
# FIGURA 4 · Ablacion por familia de variables
# ===================================================================
ab = pd.read_csv(f'{OUT}ablacion.csv')
ab = ab[ab.quita != '(ninguna)'].copy()
ab['familia'] = (ab.quita.str.replace('sin ', '', regex=False)
                 .map({'base':'Base del proceso','sup':'Superintendencia','com':'Comunidades',
                       'idx':'Índices compuestos','ent':'Entidad','cob':'Co-licitación',
                       'red':'Red bipartita','cri':'CRI'}).fillna(ab.quita))
ab = ab.sort_values('caida')
fig, ax = plt.subplots(figsize=(6.3, 0.34*len(ab) + 0.9))
yy = np.arange(len(ab))
ax.barh(yy, ab.caida, height=0.6, color=[AZUL if v > 0 else NARANJA for v in ab.caida], zorder=2)
ax.axvline(0, color=TINTA2, lw=1.0, zorder=3)
for i, v in enumerate(ab.caida):
    ax.text(v + (0.002 if v > 0 else -0.002), yy[i], c(v, 3), va='center',
            ha='left' if v > 0 else 'right', fontsize=7.5, color=TINTA2)
ax.set_yticks(yy); ax.set_yticklabels(ab.familia, fontsize=8); ax.tick_params(axis='y', length=0)
ax.set_xlabel('Caída del percentil mediano al retirar la familia completa')
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:+.2f}'.replace('.', ',')))
ax.set_xlim(ab.caida.min()-0.025, ab.caida.max()+0.025)
guarda(fig, 'fig4_ablacion', ab[['quita','familia','n_vars','mediana_percentil','lift_decil','caida']])

# ===================================================================
# FIGURA 5 · Cuantos conglomerados etiquetados hacen falta
# ===================================================================
# Experimento controlado sobre el objetivo B: mismos datos, misma etiqueta, mismo universo.
# Lo unico que cambia es cuantos conglomerados positivos se "conocen". Los no seleccionados
# vuelven a la masa de no etiquetados, que es exactamente lo que pasa en la realidad: existen
# pero nadie los ha detectado todavia. El indice fijo no depende de cuantos se conozcan.
K_CURVA, R_CURVA = 25, 8
TAMANOS = [3, 5, 8, 12, 18, 25, 35, 51]

uB = (P.tiene_grupo & P.aparece_oferentes).values.astype(bool)
dB = P[uB]
XB  = dB[VARS['B']].astype(float).values
yB  = dB.y_intragrupo.values.astype(int)
giB = pd.factorize(dB.grupo_B.values)[0]
anB = dB.anio_proceso.values
pix = dB.groupby('anio_proceso')['idx_intermediacion'].rank(pct=True).values if 'idx_intermediacion' in dB.columns else None
gpB = np.unique(giB[yB == 1])
TAMANOS = sorted({min(t, len(gpB)) for t in TAMANOS if t >= 3})
print(f'Curva sobre {len(gpB)} conglomerados positivos del objetivo B; tamaños {TAMANOS}')

f5 = []; t0 = time.time()
for t in TAMANOS:
    for rep in range(R_CURVA):
        r = np.random.default_rng(SEMILLA + 7000 + t*100 + rep)
        sel = r.choice(gpB, size=t, replace=False)
        yk  = (np.isin(giB, sel) & (yB == 1)).astype(int)        # el resto vuelve a no etiquetado
        v = estadistico(XB, yk, giB, anB, min(5, t), K_CURVA, SEMILLA + 7000 + t*100 + rep)
        vi = float(np.nanmedian(pix[yk == 1])) if pix is not None else np.nan
        f5.append((t, rep, v, vi))
    hecho = [x[2] for x in f5 if x[0] == t]
    print(f'  {t:>3} conglomerados: modelo {np.nanmean(hecho):.3f}  [{time.time()-t0:.0f} s]')
d5 = pd.DataFrame(f5, columns=['n_conglomerados','repeticion','modelo','indice'])
g5 = d5.groupby('n_conglomerados').agg(m_med=('modelo','mean'), m_min=('modelo','min'),
                                       m_max=('modelo','max'), i_med=('indice','mean')).reset_index()

fig, ax = plt.subplots(figsize=(6.3, 3.2))
ax.axhline(0.5, color=GRIS, lw=0.8, ls=':', zorder=0)
ax.fill_between(g5.n_conglomerados, g5.m_min, g5.m_max, color=AZUL, alpha=0.13, lw=0, zorder=1)
ax.plot(g5.n_conglomerados, g5.m_med, color=AZUL, lw=2.0, marker='o', ms=5,
        mec='white', mew=1.2, label='Modelo aprendido', zorder=3)
ax.plot(g5.n_conglomerados, g5.i_med, color=AGUA, lw=2.0, ls='--',
        label='Índice de intermediación (no se entrena)', zorder=2)
if max(TAMANOS) > 15:
    ax.axvline(15, color=NARANJA, lw=1.0, ls='-.', zorder=1)
    ax.text(15, ax.get_ylim()[1], ' 15 = los que tiene el objetivo A',
            fontsize=7.5, color=NARANJA, va='top', ha='left')
ax.set_xlabel('Conglomerados positivos conocidos al entrenar')
ax.set_ylabel('Percentil mediano de los positivos')
ax.yaxis.set_major_formatter(COMA)
ax.set_xticks(TAMANOS); ax.set_xticklabels(TAMANOS)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2, frameon=False, fontsize=8)
guarda(fig, 'fig5_curva_conglomerados', d5)

print('\nLas cinco figuras quedaron en', FIG)
print('Cada PNG va a 300 ppp y trae al lado el CSV con los datos exactos que la generan.')
anotar('15_figuras', carpeta=FIG, k_curva=K_CURVA, r_curva=R_CURVA, tamanos=TAMANOS,
       curva=g5.round(3).to_dict('records'))

In [ ]:
pd.read_csv(f'{FIG}/fig5_curva_conglomerados.csv').groupby('n_conglomerados').agg(
    modelo_media=('modelo','mean'), modelo_min=('modelo','min'),
    modelo_max=('modelo','max'), indice=('indice','mean')).round(3)

In [ ]:
# ¿esta idx_intermediacion dentro del conjunto rezagado?
V = json.load(open(f'{OUT}vars.json', encoding='utf-8'))
for k in ['A', 'A_anticipada', 'B', 'B_anticipada']:
    print(f'{k:16s} n={len(V[k]):3d} | idx_intermediacion dentro: '
          f'{"idx_intermediacion" in V[k]}')
print('\nLas 46 variables del conjunto anticipada de A:')
for v in sorted(V['A_anticipada']):
    print('  ', v)